In [ ]:
# ============================================================
# ETAPA 1 - CONFIGURACAO
# OBJETIVO: definir ambiente, caminhos, versao e parametros globais.
# ENTRADAS: estrutura de pastas do projeto.
# PODE ALTERAR: caminhos e metadados de execucao.
# DEVE PRESERVAR: regras de negocio e calculos do Solver.
# SAIDAS: constantes e utilitarios globais.
# VALIDACAO ESPERADA: arquivos oficiais localizados sem caminho fixo de usuario.
# ============================================================

from pathlib import Path
from collections import defaultdict, deque
from datetime import datetime
import time
import re
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter

try:
    from functions import *
except Exception:
    pass

PIPELINE_VERSION = "v66_4"
OUTPUT_SOLVER_FILENAME = f"bd_SOLVER_{PIPELINE_VERSION}.xlsx"
ESTRUTURA_FILENAME = "estruturas.xlsx"
TOL = 1e-7
MATRIZ_MAX_RECONCILIACOES = 20
MATRIZ_MAX_PASSES_DIMENSAO = 20
DATA_HORA_EXECUCAO = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
TEMPO_TOTAL_INICIO = time.perf_counter()
TEMPOS_ETAPAS = {}

try:
    CAMINHO_EXECUCAO = Path(__file__).resolve().parent
except NameError:
    CAMINHO_EXECUCAO = Path.cwd().resolve()

def resolver_raiz_ltp(caminho_inicial):
    candidatos = [caminho_inicial, caminho_inicial.parent, caminho_inicial.parent.parent]
    for candidato in candidatos:
        if (candidato / "01_INPUT").is_dir() and (candidato / "02_OUTPUT").is_dir():
            return candidato
    return caminho_inicial

RAIZ_LTP = resolver_raiz_ltp(CAMINHO_EXECUCAO)
PASTA_INPUT = RAIZ_LTP / "01_INPUT"
PASTA_OUTPUT = RAIZ_LTP / "02_OUTPUT"
if CAMINHO_EXECUCAO.name.upper() == "02_OUTPUT":
    PASTA_OUTPUT = CAMINHO_EXECUCAO
    PASTA_INPUT = CAMINHO_EXECUCAO.parent / "01_INPUT"

INPUT_FILE = PASTA_OUTPUT / "bd_LTP_NEC_SOLVER.xlsx"
OUTPUT_SOLVER_FILE = PASTA_OUTPUT / OUTPUT_SOLVER_FILENAME
ESTRUTURA_FILE = PASTA_INPUT / ESTRUTURA_FILENAME
if not INPUT_FILE.exists():
    candidatos = sorted(CAMINHO_EXECUCAO.glob("bd_LTP_NEC_SOLVER*.xlsx"))
    if candidatos:
        INPUT_FILE = candidatos[-1]

def iniciar_etapa(nome):
    TEMPOS_ETAPAS[nome] = -time.perf_counter()

def finalizar_etapa(nome):
    TEMPOS_ETAPAS[nome] = TEMPOS_ETAPAS.get(nome, 0.0) + time.perf_counter()
    return TEMPOS_ETAPAS[nome]

GUIAS_EXPORTAR = {
    "03_PLANO_PRODUCAO": True,
    "04_AUDITORIA_MATRIZ_CORTES": True,
}

print("=" * 64)
print(f" KRONA - SOLVER LTP {PIPELINE_VERSION}")
print(" Baseline funcional: V66 | Reconstrucao controlada")
print("=" * 64)


In [ ]:
# ============================================================
# ETAPA 2 - LEITURA E PREPARACAO
# OBJETIVO: ler a LTP e validar o contrato minimo de entrada.
# ENTRADAS: bd_LTP_NEC_SOLVER.xlsx e estruturas.xlsx.
# PODE ALTERAR: tipos e normalizacao tecnica dos campos.
# DEVE PRESERVAR: valores de negocio recebidos da LTP.
# SAIDAS: bd_entrada_completa e estrutura_produto.
# VALIDACAO ESPERADA: MES_REF valido, MESMA_REG/CURVAS propagaveis e chaves existentes.
# ============================================================

iniciar_etapa("LEITURA_PREPARACAO")
def to_num(s):
    return pd.to_numeric(s, errors="coerce").fillna(0)


def ensure_columns(df: pd.DataFrame, cols, default=0):
    for c in cols:
        if c not in df.columns:
            df[c] = default
    return df


def ajustar_nec_lote_embalagem(nec, lote_min, qtd_emb):
    """Aplica as regras industriais diretamente na NEC, de forma vetorizada.

    Regras:
    - NEC <= 0 permanece 0;
    - NEC positiva abaixo do LOTE_MIN sobe para o LOTE_MIN;
    - depois, quando QTD_EMB > 0, arredonda para cima para o próximo múltiplo válido.
    """
    nec = pd.to_numeric(nec, errors="coerce").fillna(0.0)
    lote_min = pd.to_numeric(lote_min, errors="coerce").fillna(0.0).clip(lower=0.0)
    qtd_emb = pd.to_numeric(qtd_emb, errors="coerce").fillna(0.0).clip(lower=0.0)

    nec_ajustada = nec.clip(lower=0.0)
    mask_pos = nec_ajustada > 0
    nec_ajustada = nec_ajustada.where(~mask_pos, np.maximum(nec_ajustada, lote_min))

    mask_emb = mask_pos & (qtd_emb > 0)
    nec_ajustada.loc[mask_emb] = np.ceil((nec_ajustada.loc[mask_emb] - 1e-12) / qtd_emb.loc[mask_emb]) * qtd_emb.loc[mask_emb]
    return nec_ajustada


def is_valid_alt(s):
    """Converte a coluna ALTERNATIVA_VALIDA para booleano de forma segura.

    Aceita booleanos reais e textos comuns como TRUE, SIM, YES, 1, VALIDA e ALTERNATIVA_VALIDA.
    Qualquer valor vazio ou diferente disso vira False.
    """
    if isinstance(s, pd.Series):
        if str(s.dtype).lower() == "bool":
            return s.fillna(False)
        txt = s.fillna(False).astype(str).str.strip().str.upper()
        return txt.isin(["TRUE", "VERDADEIRO", "SIM", "YES", "Y", "1", "VALIDA", "VÁLIDA", "ALTERNATIVA_VALIDA"])
    return bool(s)


def normalizar_codigo_serie(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
         .fillna("")
         .str.strip()
         .str.replace(r"\.0$", "", regex=True)
    )


def validar_contrato_entrada(df):
    obrigatorias = [
        "MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "UNID_PROD", "UNID_FAT",
        "MESMA_REG", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER",
        "QTD_EMB", "LOTE_MIN", "CURVA_ABC", "CURVA_123"
    ]
    faltantes = [c for c in obrigatorias if c not in df.columns]
    if faltantes:
        raise RuntimeError(f"CONTRATO_ENTRADA_COLUNAS_AUSENTES: {faltantes}")
    mesma = df["MESMA_REG"].astype(str).str.strip().str.upper()
    invalidos = sorted(mesma[~mesma.isin(["SIM", "NAO"])].dropna().unique().tolist())
    if invalidos:
        raise RuntimeError(f"MESMA_REG_INVALIDA: {invalidos[:10]}")
    if pd.to_datetime(df["MES_REF"], errors="coerce").isna().any():
        raise RuntimeError("MES_REF_INVALIDO_NA_ENTRADA")

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Arquivo de entrada nao encontrado: {INPUT_FILE}")

bd_entrada_completa = pd.read_excel(
    INPUT_FILE,
    dtype={"COD_PROD": str, "ID_PROD_UNID_FAT": str, "UNID_PROD": str, "UNID_FAT": str},
)
bd_entrada_completa["MES_REF"] = pd.to_datetime(bd_entrada_completa["MES_REF"], errors="coerce").dt.normalize()
bd_entrada_completa["COD_PROD"] = normalizar_codigo_serie(bd_entrada_completa["COD_PROD"])
bd_entrada_completa["UNID_PROD"] = bd_entrada_completa["UNID_PROD"].astype("string").fillna("").str.strip()
bd_entrada_completa["UNID_FAT"] = bd_entrada_completa["UNID_FAT"].astype("string").fillna("").str.strip()
bd_entrada_completa["MESMA_REG"] = bd_entrada_completa["MESMA_REG"].astype(str).str.strip().str.upper()
validar_contrato_entrada(bd_entrada_completa)
periodos = [pd.Timestamp(x) for x in sorted(bd_entrada_completa["MES_REF"].dropna().unique().tolist())]
if not periodos:
    raise RuntimeError("PROCESSAMENTO_MENSAL_SEM_PERIODOS_VALIDOS")

finalizar_etapa("LEITURA_PREPARACAO")
print(f"[01/10] OK - Entrada carregada: {len(bd_entrada_completa):,} linhas | {len(periodos)} periodo(s)")


In [ ]:
# ============================================================
# ETAPA 3 - CALCULO DA NEC
# OBJETIVO: formar a necessidade da V66 sem alterar sua regra vigente.
# ENTRADAS: campos LTP, estoque, LIMIT_PCS, LOTE_MIN e QTD_EMB.
# PODE ALTERAR: NEC e campos auxiliares derivados.
# DEVE PRESERVAR: componentes originais da LTP.
# SAIDAS: base mensal com NEC calculada.
# VALIDACAO ESPERADA: NEC nao negativa e LIMIT_PCS preservado.
# ============================================================

def calcular_nec_pcs_se_necessario(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula NEC_PCS somente se a coluna não existir na base."""
    df = df.copy()
    if "NEC_PCS" in df.columns:
        df = ensure_columns(df, ["LOTE_MIN", "QTD_EMB"], 0)
        df["NEC_PCS"] = ajustar_nec_lote_embalagem(df["NEC_PCS"], df["LOTE_MIN"], df["QTD_EMB"])
        if "NEC_ESTRUTURA_PAI_PCS" not in df.columns:
            df["NEC_DIRETA_BRUTA_PCS"] = df["NEC_PCS"]
            df["NEC_ESTRUTURA_PAI_PCS"] = df["NEC_PCS"]
        else:
            df["NEC_ESTRUTURA_PAI_PCS"] = ajustar_nec_lote_embalagem(df["NEC_ESTRUTURA_PAI_PCS"], df["LOTE_MIN"], df["QTD_EMB"])
        return df

    required = [
        "LTP_CART_ARR_MES_ANT", "LTP_CART_MES_ATUAL", "LTP_SALDO_PREV_PCS", "LTP_EST_SEG_PCS",
        "LTP_EST_INI_PCS", "LTP_EST_TRANS_PCS", "ORI_TOT_PCS", "TRIANG_TOT_PCS",
        "LTP_SALDO_PREV_PROX_MES_PCS", "LTP_COMP_NEC_PCS", "LIMIT_PCS", "LOTE_MIN", "QTD_EMB"
    ]
    df = ensure_columns(df, required, 0)
    for c in required:
        df[c] = to_num(df[c])

    # Demanda bruta antes de deduzir estoque/ORI/transito.
    # Esta demanda é usada como base da explosão da estrutura, seguindo o comportamento do motor antigo
    # que explodia componentes antes de o corte/solver eliminar itens por estoque/capacidade.
    demanda_bruta = (
        df["LTP_CART_ARR_MES_ANT"] + df["LTP_CART_MES_ATUAL"] + df["LTP_SALDO_PREV_PCS"] + df["LTP_EST_SEG_PCS"]
    )
    mesma_reg_nao = df.get("MESMA_REG", "SIM").astype(str).str.upper().eq("NAO")
    demanda_bruta = np.where(mesma_reg_nao, demanda_bruta + df["LTP_SALDO_PREV_PROX_MES_PCS"], demanda_bruta)
    demanda_bruta = pd.Series(demanda_bruta, index=df.index).clip(lower=0)
    demanda_bruta = demanda_bruta + df["LTP_COMP_NEC_PCS"]
    demanda_bruta = np.maximum(demanda_bruta, df["LIMIT_PCS"])

    # Demanda líquida para o solver: deduz estoque/transito/ORI/triangulação.
    base_comum = (
        df["LTP_CART_ARR_MES_ANT"] + df["LTP_CART_MES_ATUAL"] + df["LTP_SALDO_PREV_PCS"] + df["LTP_EST_SEG_PCS"]
        - df["LTP_EST_INI_PCS"] - df["LTP_EST_TRANS_PCS"] - df["ORI_TOT_PCS"] - df["TRIANG_TOT_PCS"]
    )
    nec = np.where(mesma_reg_nao, base_comum + df["LTP_SALDO_PREV_PROX_MES_PCS"], base_comum)
    nec = pd.Series(nec, index=df.index).clip(lower=0)
    nec = nec + df["LTP_COMP_NEC_PCS"]
    nec = np.maximum(nec, df["LIMIT_PCS"])


    df["NEC_PCS"] = ajustar_nec_lote_embalagem(pd.Series(nec, index=df.index).fillna(0), df["LOTE_MIN"], df["QTD_EMB"])

    # Demanda bruta calculada para auditoria e para exceções de explosão estrutural.
    # Por padrão, a explosão usa a NEC líquida do pai (NEC_PCS).
    # Exceções de retorno de componente/transferência são ativadas depois que a estrutura é conhecida.
    nec_estrutura_bruta = demanda_bruta.copy()
    df["NEC_DIRETA_BRUTA_PCS"] = pd.Series(nec_estrutura_bruta, index=df.index).fillna(0)
    df["NEC_ESTRUTURA_PAI_PCS"] = df["NEC_PCS"]
    return df


def preparar_base(df: pd.DataFrame):
    df = df.copy()
    for c in ["PRIOR_MATPAR", "PRIOR_ROT", "PCS_HORA", "HOR_REC", "HOR_FER", "NEC_PCS"]:
        if c in df.columns:
            df[c] = to_num(df[c])
    df["HOR_CAP"] = df[["HOR_REC", "HOR_FER"]].min(axis=1)
    mes_ref = df["MES_REF"].min()
    bd_mes = df[df["MES_REF"].eq(mes_ref)].copy()
    return bd_mes


def calcular_nec_periodo(bd_periodo):
    bd = bd_periodo.copy()
    bd["COD_PROD"] = normalizar_codigo_serie(bd["COD_PROD"])
    bd["UNID_PROD"] = bd["UNID_PROD"].astype("string").fillna("").str.strip()
    bd["UNID_FAT"] = bd["UNID_FAT"].astype("string").fillna("").str.strip()
    bd = calcular_nec_pcs_se_necessario(bd)
    bd_mes = preparar_base(bd)
    if (to_num(bd_mes["NEC_PCS"]) < -TOL).any():
        raise RuntimeError("NEC_NEGATIVA_APOS_CALCULO")
    return bd_mes

print("[02/10] OK - Regras de formacao da NEC carregadas")


In [ ]:
# ============================================================
# ETAPA 4 - ESTRUTURA / PA -> PI
# OBJETIVO: explodir estrutura pela UNID_PROD e retornar componentes por COD_INSUMO -> COD_PROD e UNID_PROD -> UNID_FAT.
# ENTRADAS: NEC liquida dos PAs, estrutura e estoque global dos componentes.
# PODE ALTERAR: COMP_NEC/NEC consolidada por efeito estrutural.
# DEVE PRESERVAR: chave de retorno validada e estoque unico do PI.
# SAIDAS: estrutura_explodida, problemas_estrutura e nec_consolidada_solver.
# VALIDACAO ESPERADA: sem duplicacao de NEC e sem quebra da chave de retorno.
# ============================================================

def ler_estrutura_produto(caminho_estrutura: Path) -> pd.DataFrame:
    if not caminho_estrutura.exists():
        print("Arquivo de estrutura não encontrado. A explosão será ignorada:", caminho_estrutura)
        return pd.DataFrame()

    estrutura = pd.read_excel(
        caminho_estrutura,
        sheet_name="estruturas",
        dtype={"cod_prod_acabado": str, "cod_insumo": str, "empresa": str}
    )

    required = ["empresa", "cod_prod_acabado", "cod_insumo", "qtd_utilizada_pcs"]
    missing = [c for c in required if c not in estrutura.columns]
    if missing:
        raise ValueError(f"Colunas obrigatórias ausentes na estrutura: {missing}")

    estrutura = estrutura.copy()
    estrutura["empresa"] = estrutura["empresa"].astype("string").fillna("").str.strip()
    estrutura["cod_prod_acabado"] = normalizar_codigo_serie(estrutura["cod_prod_acabado"])
    estrutura["cod_insumo"] = normalizar_codigo_serie(estrutura["cod_insumo"])
    estrutura["qtd_utilizada_pcs"] = to_num(estrutura["qtd_utilizada_pcs"])
    estrutura = estrutura[
        (estrutura["empresa"] != "")
        & (estrutura["cod_prod_acabado"] != "")
        & (estrutura["cod_insumo"] != "")
    ].copy()

    estrutura = estrutura.drop_duplicates(["empresa", "cod_prod_acabado", "cod_insumo"], keep="first")
    return estrutura


def calcular_estoque_componentes_pos_demanda_direta(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula estoque disponível para a explosão após consumir a demanda direta/bruta do próprio item.

    Mantém a lógica do pipeline antigo: estoque total = estoque inicial + trânsito + ORI + triangulação.
    Depois abate a demanda bruta do item, sem descontar estoque nessa demanda, para evitar usar na estrutura
    o estoque que já estaria comprometido com a necessidade direta do próprio produto.
    """
    df = df.copy()
    required = [
        "LTP_EST_INI_PCS", "LTP_EST_TRANS_PCS", "ORI_TOT_PCS", "TRIANG_TOT_PCS",
        "LTP_CART_ARR_MES_ANT", "LTP_CART_MES_ATUAL", "LTP_SALDO_PREV_PCS",
        "LTP_SALDO_PREV_PROX_MES_PCS", "LTP_EST_SEG_PCS", "LIMIT_PCS", "LOTE_MIN", "QTD_EMB"
    ]
    df = ensure_columns(df, required, 0)
    for c in required:
        df[c] = to_num(df[c])

    df["COD_PROD"] = normalizar_codigo_serie(df["COD_PROD"])
    df["UNID_PROD"] = df["UNID_PROD"].astype("string").fillna("").str.strip()
    if "UNID_FAT" not in df.columns:
        df["UNID_FAT"] = df["UNID_PROD"]
    df["UNID_FAT"] = df["UNID_FAT"].astype("string").fillna("").str.strip()
    tipo_prod = df.get("TIPO_PROD", "").astype(str).str.upper()
    mesma_reg = df.get("MESMA_REG", "SIM").astype(str).str.upper()

    estoque_total = (
        df["LTP_EST_INI_PCS"]
        + df["LTP_EST_TRANS_PCS"]
        + df["ORI_TOT_PCS"]
        + df["TRIANG_TOT_PCS"]
    )

    demanda_base = (
        df["LTP_CART_ARR_MES_ANT"]
        + df["LTP_CART_MES_ATUAL"]
        + df["LTP_SALDO_PREV_PCS"]
        + df["LTP_EST_SEG_PCS"]
    )
    mask_mesma_reg_nao = mesma_reg.eq("NAO")
    demanda_base = demanda_base.where(~mask_mesma_reg_nao, demanda_base + df["LTP_SALDO_PREV_PROX_MES_PCS"])

    demanda_bruta = np.maximum(demanda_base, df["LIMIT_PCS"])

    df["ESTOQUE_TOTAL_PCS"] = estoque_total
    df["DEMANDA_BRUTA_DIRETA_PCS"] = np.maximum(demanda_bruta, 0)
    df["ESTOQUE_DISP_EXPLOSAO_PCS"] = np.maximum(df["ESTOQUE_TOTAL_PCS"] - df["DEMANDA_BRUTA_DIRETA_PCS"], 0)

    # Regra herdada do motor de cortes:
    # o estoque disponível para a explosão é localizado pela unidade de faturamento/destino.
    # Isso permite que uma demanda Nordeste do pai gere necessidade em uma linha do componente
    # produzida em Matriz, desde que a linha do componente tenha UNID_FAT = Nordeste.
    estoque = (
        df.groupby(["COD_PROD", "UNID_FAT"], as_index=False)
        .agg(
            ESTOQUE_TOTAL_PCS=("ESTOQUE_TOTAL_PCS", "max"),
            DEMANDA_BRUTA_DIRETA_PCS=("DEMANDA_BRUTA_DIRETA_PCS", "max"),
            ESTOQUE_DISP_EXPLOSAO_PCS=("ESTOQUE_DISP_EXPLOSAO_PCS", "max")
        )
        .rename(columns={"UNID_FAT": "UNID_PROD"})
    )
    return estoque


def explodir_estrutura_multinivel_com_estoque(
    bd_estrutura: pd.DataFrame,
    bd_demanda_raiz: pd.DataFrame,
    bd_estoque_componentes: pd.DataFrame,
    bd_mapa_componentes: pd.DataFrame = None,
    max_nivel: int = 30
):
    cols = [
        "MES_REF", "UNID_PROD", "ID_PROD_UNID_FAT_ORIGEM", "COD_PROD_ORIGEM",
        "NEC_RAIZ_PCS", "COD_PROD_ACAB", "COD_INSUMO", "QTD_UTIL_PCS",
        "NEC_GERADA_BRUTA_PCS", "ESTOQUE_COMP_ANTES_PCS", "ESTOQUE_CONSUMIDO_PCS",
        "NEC_GERADA_LIQ_PCS", "ESTOQUE_COMP_DEPOIS_PCS", "NIVEL", "TRILHA"
    ]
    if bd_estrutura.empty or bd_demanda_raiz.empty:
        return pd.DataFrame(columns=cols), pd.DataFrame(columns=["UNID_PROD", "COD_PROD_ORIGEM", "TRILHA", "MOTIVO"])

    estrutura_idx = defaultdict(lambda: defaultdict(list))
    for emp, pai, filho, qtd in bd_estrutura[["empresa", "cod_prod_acabado", "cod_insumo", "qtd_utilizada_pcs"]].itertuples(index=False, name=None):
        estrutura_idx[str(emp)][str(pai)].append((str(filho), float(qtd or 0)))

    estoque_dict = {
        (str(unid), str(cod)): float(est or 0)
        for cod, unid, est in bd_estoque_componentes[["COD_PROD", "UNID_PROD", "ESTOQUE_DISP_EXPLOSAO_PCS"]].itertuples(index=False, name=None)
    }

    # Regra oficial de chaveamento da explosão/retorno:
    # A estrutura é explodida na UNID_PROD corrente e essa unidade é preservada
    # na necessidade gerada. No retorno ao LTP, o casamento é:
    #   COD_INSUMO -> COD_PROD
    #   UNID_PROD  -> UNID_FAT
    # Para continuar a explosão multinível, localiza-se o componente no LTP
    # por COD_PROD + UNID_FAT e usa-se a UNID_PROD dessa linha no próximo nível.
    if bd_mapa_componentes is None or bd_mapa_componentes.empty:
        mapa_comp = pd.DataFrame(columns=["COD_PROD", "UNID_FAT", "UNID_PROD"])
    else:
        mapa_comp = bd_mapa_componentes.copy()
    for _c in ["COD_PROD", "UNID_FAT", "UNID_PROD"]:
        if _c not in mapa_comp.columns:
            mapa_comp[_c] = ""
        mapa_comp[_c] = mapa_comp[_c].astype("string").fillna("").str.strip()
    mapa_comp["COD_PROD"] = normalizar_codigo_serie(mapa_comp["COD_PROD"])
    mapa_comp = mapa_comp.drop_duplicates(["COD_PROD", "UNID_FAT"], keep="first")
    comp_por_fat = {
        (str(cod), str(unid_fat)): str(unid_prod)
        for cod, unid_fat, unid_prod
        in mapa_comp[["COD_PROD", "UNID_FAT", "UNID_PROD"]].itertuples(index=False, name=None)
    }

    def resolver_unidade_prod_componente(cod_filho, unid_prod_explosao):
        cod_filho = str(cod_filho)
        unid_prod_explosao = str(unid_prod_explosao or "").strip()
        return comp_por_fat.get((cod_filho, unid_prod_explosao), unid_prod_explosao)

    resultados = []
    problemas = []

    ordem_cols = ["MES_REF", "UNID_PROD", "COD_PROD", "ID_PROD_UNID_FAT"]
    bd_demanda_raiz_ord = bd_demanda_raiz.sort_values([c for c in ordem_cols if c in bd_demanda_raiz.columns]).copy()

    for row in bd_demanda_raiz_ord[["MES_REF", "UNID_PROD", "UNID_FAT_ORIGEM", "ID_PROD_UNID_FAT", "COD_PROD", "NEC_PCS"]].itertuples(index=False, name=None):
        mes_ref, unid_estrutura, unid_fat_origem, id_origem, cod_origem, nec_raiz = row
        emp = str(unid_estrutura)
        unid_fat_origem = str(unid_fat_origem or emp).strip()
        root = str(cod_origem)
        nec_raiz = float(nec_raiz or 0)
        if nec_raiz <= TOL:
            continue
        estrutura_empresa = estrutura_idx.get(emp, {})
        if root not in estrutura_empresa:
            continue

        # Na fila entra a necessidade líquida do pai.
        # A fila carrega a unidade estrutural do nó para localizar seus filhos, mantendo separada a UNID_FAT da origem.
        fila = deque([(emp, root, 0, nec_raiz, root, {root})])
        while fila:
            emp_no, pai, nivel, nec_pai_liq, trilha, caminho = fila.popleft()
            if nivel >= max_nivel:
                problemas.append((emp_no, root, trilha, "MAX_NIVEL_ATINGIDO"))
                continue

            estrutura_empresa_no = estrutura_idx.get(str(emp_no), {})
            for filho, qtd_util in estrutura_empresa_no.get(pai, []):
                nivel_filho = nivel + 1
                trilha_filho = f"{trilha} -> {filho}"
                ciclo = filho in caminho
                nec_bruta = float(nec_pai_liq or 0) * float(qtd_util or 0)

                if ciclo:
                    problemas.append((emp_no, root, trilha_filho, "CICLO_ESTRUTURA"))
                    continue
                if nec_bruta <= TOL:
                    continue

                # A linha explodida preserva a UNID_PROD que gerou a necessidade.
                # No retorno ao LTP: UNID_PROD -> UNID_FAT e COD_INSUMO -> COD_PROD.
                unid_retorno = str(emp_no).strip()
                unid_prod_componente = resolver_unidade_prod_componente(filho, unid_retorno)
                chave_estoque = (unid_retorno, str(filho))
                estoque_antes = float(estoque_dict.get(chave_estoque, 0.0))
                estoque_consumido = min(estoque_antes, nec_bruta)
                nec_liq = max(nec_bruta - estoque_consumido, 0.0)
                estoque_depois = max(estoque_antes - estoque_consumido, 0.0)
                estoque_dict[chave_estoque] = estoque_depois

                resultados.append((
                    mes_ref, unid_retorno, id_origem, root, nec_raiz, pai, filho, qtd_util,
                    nec_bruta, estoque_antes, estoque_consumido, nec_liq, estoque_depois,
                    nivel_filho, trilha_filho
                ))

                # Só explode abaixo do componente se ainda existe necessidade líquida dele.
                # Para localizar os filhos do componente, usa a unidade produtiva do componente.
                if nec_liq > TOL:
                    fila.append((unid_prod_componente, filho, nivel_filho, nec_liq, trilha_filho, caminho | {filho}))

    bd_explodida = pd.DataFrame.from_records(resultados, columns=cols)
    bd_problemas = pd.DataFrame.from_records(problemas, columns=["UNID_PROD", "COD_PROD_ORIGEM", "TRILHA", "MOTIVO"])
    return bd_explodida, bd_problemas


def consolidar_necessidade_com_estrutura(bd_mes: pd.DataFrame, bd_estrutura_explodida: pd.DataFrame):
    """Consolida demanda direta e demanda de componente seguindo o motor antigo.

    Regra principal:
    - a explosão da estrutura nasce pela unidade produtiva do pai;
    - a necessidade líquida do componente volta para o LTP pelo chaveamento definido pelo cliente;
    - a chave de retorno é:
        componente.COD_PROD = COD_INSUMO explodido
        componente.UNID_FAT = UNID_PROD registrada na explosão
    - o solver passa a trabalhar com duas naturezas de demanda:
        NEC_DIRETA_PCS = demanda própria do item;
        NEC_COMP_PCS   = demanda líquida gerada por pais/componentes;
        NEC_PCS        = NEC_DIRETA_PCS + NEC_COMP_PCS.
    """
    bd_mes_out = bd_mes.copy()
    if "UNID_FAT" not in bd_mes_out.columns:
        bd_mes_out["UNID_FAT"] = bd_mes_out["UNID_PROD"]
    bd_mes_out["COD_PROD"] = normalizar_codigo_serie(bd_mes_out["COD_PROD"])
    bd_mes_out["UNID_PROD"] = bd_mes_out["UNID_PROD"].astype("string").fillna("").str.strip()
    bd_mes_out["UNID_FAT"] = bd_mes_out["UNID_FAT"].astype("string").fillna("").str.strip()

    bd_mes_out["NEC_PCS_ORIGINAL"] = to_num(bd_mes_out["NEC_PCS"])
    bd_mes_out["NEC_DIRETA_PCS"] = bd_mes_out["NEC_PCS_ORIGINAL"]

    if bd_estrutura_explodida is None or bd_estrutura_explodida.empty:
        nec_comp_retorno = pd.DataFrame(columns=[
            "MES_REF", "UNID_FAT", "COD_PROD",
            "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_COMP_PCS"
        ])
    else:
        nec_comp_retorno = (
            bd_estrutura_explodida
            .groupby(["MES_REF", "UNID_PROD", "COD_INSUMO"], as_index=False)
            .agg(
                NEC_EXPLODIDA_BRUTA_PCS=("NEC_GERADA_BRUTA_PCS", "sum"),
                ESTOQUE_CONSUMIDO_EXPLOSAO_PCS=("ESTOQUE_CONSUMIDO_PCS", "sum"),
                NEC_COMP_PCS=("NEC_GERADA_LIQ_PCS", "sum")
            )
            .rename(columns={"UNID_PROD": "UNID_FAT", "COD_INSUMO": "COD_PROD"})
        )
        nec_comp_retorno["COD_PROD"] = normalizar_codigo_serie(nec_comp_retorno["COD_PROD"])
        nec_comp_retorno["UNID_FAT"] = nec_comp_retorno["UNID_FAT"].astype("string").fillna("").str.strip()

    bd_mes_out = bd_mes_out.merge(
        nec_comp_retorno,
        on=["MES_REF", "UNID_FAT", "COD_PROD"],
        how="left"
    )

    for c in ["NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_COMP_PCS"]:
        bd_mes_out[c] = to_num(bd_mes_out[c])

    # Alias mantido para compatibilidade com guias antigas.
    bd_mes_out["NEC_EXPLODIDA_PCS"] = bd_mes_out["NEC_COMP_PCS"]
    bd_mes_out["LTP_COMP_NEC_PCS"] = bd_mes_out["NEC_COMP_PCS"]
    bd_mes_out["NEC_PCS_CONSOLIDADA"] = bd_mes_out["NEC_DIRETA_PCS"] + bd_mes_out["NEC_COMP_PCS"]
    bd_mes_out = ensure_columns(bd_mes_out, ["LOTE_MIN", "QTD_EMB"], 0)
    bd_mes_out["NEC_PCS_CONSOLIDADA"] = ajustar_nec_lote_embalagem(bd_mes_out["NEC_PCS_CONSOLIDADA"], bd_mes_out["LOTE_MIN"], bd_mes_out["QTD_EMB"])
    bd_mes_out["NEC_PCS"] = bd_mes_out["NEC_PCS_CONSOLIDADA"]

    chave_item = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "UNID_PROD", "UNID_FAT"]
    extras = ["DESC_PROD", "TIPO_PROD"]
    agg = {
        "NEC_DIRETA_PCS": "max",
        "NEC_COMP_PCS": "max",
        "NEC_EXPLODIDA_BRUTA_PCS": "max",
        "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS": "max",
        "NEC_EXPLODIDA_PCS": "max",
        "NEC_PCS_CONSOLIDADA": "max",
    }
    for c in extras:
        if c in bd_mes_out.columns:
            agg[c] = "first"
    nec_consolidada = (
        bd_mes_out.groupby([c for c in chave_item if c in bd_mes_out.columns], as_index=False)
        .agg(agg)
    )
    nec_consolidada["TEM_ROTEIRO_NO_SOLVER"] = nec_consolidada["ID_PROD_UNID_FAT"].notna()

    return bd_mes_out, nec_consolidada


def processar_estrutura_periodo(bd_mes):
    estrutura_produto = ler_estrutura_produto(ESTRUTURA_FILE)
    estoque_componentes = calcular_estoque_componentes_pos_demanda_direta(bd_mes)

    bd_mes = bd_mes.copy()
    bd_mes["UNID_ESTRUTURA"] = bd_mes["UNID_PROD"].astype("string").fillna("").str.strip()
    bd_mes["UNID_FAT_ORIGEM"] = bd_mes["UNID_FAT"].where(
        bd_mes["UNID_FAT"].astype("string").fillna("").str.strip().ne(""), bd_mes["UNID_PROD"]
    ).astype("string").fillna("").str.strip()

    demanda_raiz = (
        bd_mes.sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])
        .groupby(["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD"], as_index=False)
        .agg(
            UNID_PROD=("UNID_ESTRUTURA", "first"),
            UNID_FAT_ORIGEM=("UNID_FAT_ORIGEM", "first"),
            NEC_PCS=("NEC_ESTRUTURA_PAI_PCS", "max"),
        )
    )
    demanda_raiz = demanda_raiz[demanda_raiz["NEC_PCS"] > TOL].copy()

    mapa_retorno = (
        bd_mes.sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])
        .groupby(["COD_PROD", "UNID_FAT"], as_index=False)
        .agg(UNID_PROD=("UNID_PROD", "first"))
    )

    estrutura_explodida, problemas_estrutura = explodir_estrutura_multinivel_com_estoque(
        estrutura_produto, demanda_raiz, estoque_componentes, mapa_retorno, max_nivel=30
    )
    bd_mes, nec_consolidada = consolidar_necessidade_com_estrutura(bd_mes, estrutura_explodida)
    return bd_mes, estrutura_explodida, problemas_estrutura, nec_consolidada

print("[03/10] OK - Estrutura PA -> PI preparada")


In [ ]:
# ============================================================
# ETAPA 5 - MATRIZ DE CORTES
# OBJETIVO: reduzir somente origens cortaveis para equilibrar capacidade antes do Solver.
# ENTRADAS: estado da necessidade, rotas e mapa PA -> PI.
# PODE ALTERAR: C_ARR, C_AT, PREV_VEN, PREV_PROX e EST_SEG.
# DEVE PRESERVAR: COMP_NEC direto, EST_IN, EST_TRANS, LIMIT_PCS, ORI_* e ordem dos 9 criterios.
# SAIDAS: NEC pos-Matriz e 04_AUDITORIA_MATRIZ_CORTES.
# VALIDACAO ESPERADA: ALOC_REC -> COD_FER_UNID -> reconciliacao ate estabilizacao real.
# ============================================================

MAP_COMPONENTE_LTP = {
    "C_ARR": "LTP_CART_ARR_MES_ANT",
    "C_AT": "LTP_CART_MES_ATUAL",
    "PREV_VEN": "LTP_SALDO_PREV_PCS",
    "PREV_PROX": "LTP_SALDO_PREV_PROX_MES_PCS",
    "EST_SEG": "LTP_EST_SEG_PCS",
    "EST_IN": "LTP_EST_INI_PCS",
    "EST_TRANS": "LTP_EST_TRANS_PCS",
    "COMP_NEC": "LTP_COMP_NEC_PCS",
}

MATRIZ_CRITERIOS = [
    (1, "SIM", "EST_SEG"),
    (2, "NAO", "PREV_PROX"),
    (3, "NAO", "EST_SEG"),
    (4, "SIM", "PREV_VEN"),
    (5, "NAO", "PREV_VEN"),
    (6, "SIM", "C_AT"),
    (7, "NAO", "C_AT"),
    (8, "SIM", "C_ARR"),
    (9, "NAO", "C_ARR"),
]

COLS_SNAPSHOT_MATRIZ = [
    "C_ARR", "C_AT", "PREV_VEN", "PREV_PROX", "EST_SEG",
    "EST_IN", "EST_TRANS", "COMP_NEC", "NEC_PCS", "LIMIT_PCS", "LOTE_MIN", "QTD_EMB",
]

def recalcular_nec_estado_matriz(estado):
    out = estado.copy()
    for c in [
        "C_ARR", "C_AT", "PREV_VEN", "PREV_PROX", "EST_SEG", "EST_IN", "EST_TRANS", "COMP_NEC",
        "ORI_TOT_PCS", "TRIANG_TOT_PCS", "LIMIT_PCS", "LOTE_MIN", "QTD_EMB"
    ]:
        if c not in out.columns:
            out[c] = 0.0
        out[c] = to_num(out[c])
    mesma_reg_nao = out["MESMA_REG"].astype(str).str.strip().str.upper().eq("NAO")
    base = (
        out["C_ARR"] + out["C_AT"] + out["PREV_VEN"] + out["EST_SEG"]
        - out["EST_IN"] - out["EST_TRANS"] - out["ORI_TOT_PCS"] - out["TRIANG_TOT_PCS"]
    )
    base = np.where(mesma_reg_nao, base + out["PREV_PROX"], base)
    nec = pd.Series(base, index=out.index).clip(lower=0.0) + out["COMP_NEC"]
    nec = np.maximum(nec, out["LIMIT_PCS"])
    out["NEC_PCS"] = ajustar_nec_lote_embalagem(pd.Series(nec, index=out.index), out["LOTE_MIN"], out["QTD_EMB"])
    return out

def preparar_estado_matriz(bd_mes):
    base = bd_mes.copy()
    base["MESMA_REG"] = base["MESMA_REG"].astype(str).str.strip().str.upper()
    for curto, origem in MAP_COMPONENTE_LTP.items():
        if origem not in base.columns:
            base[origem] = 0.0
        base[curto] = to_num(base[origem]).astype(float)
    for c in ["ORI_TOT_PCS", "TRIANG_TOT_PCS", "LIMIT_PCS", "LOTE_MIN", "QTD_EMB", "NEC_PCS"]:
        if c not in base.columns:
            base[c] = 0.0
        base[c] = to_num(base[c]).astype(float)
    ordem = [c for c in ["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"] if c in base.columns]
    estado = base.sort_values(ordem).drop_duplicates("ID_PROD_UNID_FAT", keep="first").copy().reset_index(drop=True)
    estado["ORI_C_ARR"] = estado["C_ARR"]
    estado["ORI_C_AT"] = estado["C_AT"]
    estado["ORI_PREV_VEN"] = estado["PREV_VEN"]
    estado["ORI_PREV_PROX"] = estado["PREV_PROX"]
    estado["ORI_EST_SEG"] = estado["EST_SEG"]
    estado["ORI_NEC"] = estado["NEC_PCS"]
    return base, estado

def preparar_alternativas_matriz(bd_mes):
    cols = [
        "MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT",
        "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER"
    ]
    alt = bd_mes[[c for c in cols if c in bd_mes.columns]].copy()
    for c in ["PRIOR_MATPAR", "PRIOR_ROT", "PCS_HORA", "HOR_REC", "HOR_FER"]:
        alt[c] = to_num(alt[c])
    alt = alt[
        alt["ALOC_REC"].notna() & alt["COD_FER_UNID"].notna()
        & (alt["PCS_HORA"] > TOL) & (alt["HOR_REC"] > TOL) & (alt["HOR_FER"] > TOL)
    ].copy()
    return alt.drop_duplicates([
        "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID"
    ]).reset_index(drop=True)

def simular_rateio_matriz(estado, alternativas, dimensoes=("ALOC_REC", "COD_FER_UNID")):
    if alternativas.empty or estado.empty:
        return pd.Series(dtype=float), pd.DataFrame()
    alt = alternativas.copy()
    pendente = {k: float(v) for k, v in estado.set_index("ID_PROD_UNID_FAT")["NEC_PCS"].items()}
    cap_rec = alt.groupby("ALOC_REC")["HOR_REC"].max().to_dict()
    cap_fer = alt.groupby("COD_FER_UNID")["HOR_FER"].max().to_dict()
    rastros = []
    niveis = list(alt[["PRIOR_MATPAR", "PRIOR_ROT"]].drop_duplicates().sort_values(["PRIOR_MATPAR", "PRIOR_ROT"]).itertuples(index=False, name=None))
    for rodada, (pm, pr) in enumerate(niveis, start=1):
        n = alt[alt["PRIOR_MATPAR"].eq(pm) & alt["PRIOR_ROT"].eq(pr)].copy()
        n["QTD_PENDENTE_ANTES"] = n["ID_PROD_UNID_FAT"].map(pendente).fillna(0.0)
        n = n[n["QTD_PENDENTE_ANTES"] > TOL].copy()
        if n.empty:
            continue
        qtd_alt = n.groupby("ID_PROD_UNID_FAT")["PCS_HORA"].transform("count").clip(lower=1)
        n["QTD_SOLICITADA"] = n["QTD_PENDENTE_ANTES"] / qtd_alt
        n["HR_SOLICITADA"] = n["QTD_SOLICITADA"] / n["PCS_HORA"]
        fatores = []
        if "ALOC_REC" in dimensoes:
            n["CAP_REC_ANTES"] = n["ALOC_REC"].map(cap_rec).fillna(0.0)
            dem_rec = n.groupby("ALOC_REC")["HR_SOLICITADA"].transform("sum")
            n["FATOR_REC"] = np.where(dem_rec > TOL, np.minimum(1.0, n["CAP_REC_ANTES"] / dem_rec), 0.0)
            fatores.append("FATOR_REC")
        else:
            n["FATOR_REC"] = 1.0
        if "COD_FER_UNID" in dimensoes:
            n["CAP_FER_ANTES"] = n["COD_FER_UNID"].map(cap_fer).fillna(0.0)
            dem_fer = n.groupby("COD_FER_UNID")["HR_SOLICITADA"].transform("sum")
            n["FATOR_FER"] = np.where(dem_fer > TOL, np.minimum(1.0, n["CAP_FER_ANTES"] / dem_fer), 0.0)
            fatores.append("FATOR_FER")
        else:
            n["FATOR_FER"] = 1.0
        n["FATOR_MATRIZ"] = n[fatores].min(axis=1) if fatores else 1.0
        n["HR_PRODUZIDA"] = n["HR_SOLICITADA"] * n["FATOR_MATRIZ"]
        n["QTD_PRODUZIDA"] = n["HR_PRODUZIDA"] * n["PCS_HORA"]
        n["RODADA_SIM"] = rodada
        if "ALOC_REC" in dimensoes:
            for k, v in n.groupby("ALOC_REC")["HR_PRODUZIDA"].sum().items():
                cap_rec[k] = max(0.0, float(cap_rec.get(k, 0.0)) - float(v))
        if "COD_FER_UNID" in dimensoes:
            for k, v in n.groupby("COD_FER_UNID")["HR_PRODUZIDA"].sum().items():
                cap_fer[k] = max(0.0, float(cap_fer.get(k, 0.0)) - float(v))
        for k, v in n.groupby("ID_PROD_UNID_FAT")["QTD_PRODUZIDA"].sum().items():
            pendente[k] = max(0.0, float(pendente.get(k, 0.0)) - float(v))
        rastros.append(n)
    return pd.Series(pendente, dtype=float), pd.concat(rastros, ignore_index=True) if rastros else pd.DataFrame()

def gaps_por_restricao(pendente, rastro, dimensao):
    if rastro.empty or pendente.empty:
        return pd.DataFrame(columns=[dimensao, "ID_PROD_UNID_FAT", "CORTE_HR_NECESSARIO"])
    ids_gap = pendente[pendente > TOL]
    if ids_gap.empty:
        return pd.DataFrame(columns=[dimensao, "ID_PROD_UNID_FAT", "CORTE_HR_NECESSARIO"])
    fator_col = "FATOR_REC" if dimensao == "ALOC_REC" else "FATOR_FER"
    r = rastro[rastro["ID_PROD_UNID_FAT"].isin(ids_gap.index)].copy()
    r = r[(r[fator_col] < 1.0 - TOL) & (r["QTD_SOLICITADA"] > TOL)].copy()
    if r.empty:
        return pd.DataFrame(columns=[dimensao, "ID_PROD_UNID_FAT", "CORTE_HR_NECESSARIO"])
    ult = r.groupby("ID_PROD_UNID_FAT")["RODADA_SIM"].transform("max")
    r = r[r["RODADA_SIM"].eq(ult)].copy()
    r["GAP_ITEM"] = r["ID_PROD_UNID_FAT"].map(ids_gap).fillna(0.0)
    r["SHORTFALL"] = (r["QTD_SOLICITADA"] - r["QTD_PRODUZIDA"]).clip(lower=0.0)
    soma = r.groupby("ID_PROD_UNID_FAT")["SHORTFALL"].transform("sum")
    qtd = r.groupby("ID_PROD_UNID_FAT")[dimensao].transform("count").clip(lower=1)
    r["PESO_GAP"] = np.where(soma > TOL, r["SHORTFALL"] / soma, 1.0 / qtd)
    r["GAP_ATRIBUIDO_PCS"] = r["GAP_ITEM"] * r["PESO_GAP"]
    r["CORTE_HR_NECESSARIO"] = np.where(r["PCS_HORA"] > TOL, r["GAP_ATRIBUIDO_PCS"] / r["PCS_HORA"], 0.0)
    return r[[dimensao, "ID_PROD_UNID_FAT", "PCS_HORA", "CORTE_HR_NECESSARIO"]].copy()

def construir_mapa_impacto_pa_pi(estrutura_explodida, estado):
    cols = ["PA_ID", "PA_COD_PROD", "PI_ID", "PI_COD_PROD", "FATOR_ESTRUTURAL"]
    if estrutura_explodida is None or estrutura_explodida.empty:
        return pd.DataFrame(columns=cols)
    e = estrutura_explodida.copy()
    for c in ["NEC_GERADA_BRUTA_PCS", "NEC_RAIZ_PCS"]:
        e[c] = to_num(e[c])
    e["FATOR_ESTRUTURAL"] = np.divide(
        e["NEC_GERADA_BRUTA_PCS"].to_numpy(float), e["NEC_RAIZ_PCS"].to_numpy(float),
        out=np.zeros(len(e), dtype=float), where=e["NEC_RAIZ_PCS"].to_numpy(float) > TOL
    )
    e = e[e["FATOR_ESTRUTURAL"] > TOL].copy()
    pi = estado[["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "UNID_FAT"]].copy()
    pi = pi.rename(columns={"ID_PROD_UNID_FAT": "PI_ID", "COD_PROD": "PI_COD_PROD", "UNID_FAT": "UNID_PROD"})
    e = e.merge(pi, left_on=["MES_REF", "COD_INSUMO", "UNID_PROD"], right_on=["MES_REF", "PI_COD_PROD", "UNID_PROD"], how="inner")
    e["PA_ID"] = e["ID_PROD_UNID_FAT_ORIGEM"].astype(str)
    e["PA_COD_PROD"] = e["COD_PROD_ORIGEM"].astype(str)
    return e.groupby(["PA_ID", "PA_COD_PROD", "PI_ID", "PI_COD_PROD"], as_index=False)["FATOR_ESTRUTURAL"].sum()[cols]

def participantes_restricao(rastro, dimensao, chave):
    if rastro.empty:
        return pd.DataFrame(columns=["ID_PROD_UNID_FAT", "PCS_HORA"])
    r = rastro[rastro[dimensao].astype(str).eq(str(chave)) & (rastro["QTD_SOLICITADA"] > TOL)].copy()
    if r.empty:
        return pd.DataFrame(columns=["ID_PROD_UNID_FAT", "PCS_HORA"])
    r = r.sort_values(["ID_PROD_UNID_FAT", "RODADA_SIM"]).drop_duplicates("ID_PROD_UNID_FAT", keep="last")
    return r[["ID_PROD_UNID_FAT", "PCS_HORA"]].copy()

def candidatos_criterio(estado, mapa_pa_pi, participantes, mesma_reg, componente):
    if participantes.empty:
        return pd.DataFrame()
    p = participantes.copy()
    p["ID_PROD_UNID_FAT"] = p["ID_PROD_UNID_FAT"].astype(str)
    st = estado.copy()
    st["ID_PROD_UNID_FAT"] = st["ID_PROD_UNID_FAT"].astype(str)

    direto = st.merge(p, on="ID_PROD_UNID_FAT", how="inner", suffixes=("", "_ROT"))
    direto = direto[direto["MESMA_REG"].eq(mesma_reg) & (to_num(direto[componente]) > TOL)].copy()
    if not direto.empty:
        direto["COEF_HR_POR_PCA"] = np.divide(
            1.0, direto["PCS_HORA_ROT"], out=np.zeros(len(direto)), where=to_num(direto["PCS_HORA_ROT"]).to_numpy(float) > TOL
        )
        direto["TIPO_IMPACTO"] = "DIRETO"
        direto["PA_GERADOR"] = ""
        direto["PI_IMPACTADO"] = ""
        direto = direto[["ID_PROD_UNID_FAT", "COD_PROD", "UNID_FAT", componente, "COEF_HR_POR_PCA", "TIPO_IMPACTO", "PA_GERADOR", "PI_IMPACTADO"]]

    indireto = pd.DataFrame()
    if mapa_pa_pi is not None and not mapa_pa_pi.empty:
        rel = mapa_pa_pi[mapa_pa_pi["PI_ID"].astype(str).isin(set(p["ID_PROD_UNID_FAT"]))].copy()
        if not rel.empty:
            rel = rel.merge(p.rename(columns={"ID_PROD_UNID_FAT": "PI_ID", "PCS_HORA": "PCS_HORA_PI"}), on="PI_ID", how="inner")
            rel["COEF_HR_POR_PCA"] = np.divide(
                rel["FATOR_ESTRUTURAL"].to_numpy(float), rel["PCS_HORA_PI"].to_numpy(float),
                out=np.zeros(len(rel)), where=rel["PCS_HORA_PI"].to_numpy(float) > TOL
            )
            rel = rel[rel["COEF_HR_POR_PCA"] > TOL].copy()
            rel = rel.groupby("PA_ID", as_index=False).agg(
                COEF_HR_POR_PCA=("COEF_HR_POR_PCA", "sum"),
                PI_IMPACTADO=("PI_COD_PROD", lambda s: "|".join(sorted(set(map(str, s))))),
            )
            indireto = st.merge(rel, left_on="ID_PROD_UNID_FAT", right_on="PA_ID", how="inner")
            indireto = indireto[indireto["MESMA_REG"].eq(mesma_reg) & (to_num(indireto[componente]) > TOL)].copy()
            if not indireto.empty:
                indireto["TIPO_IMPACTO"] = "INDIRETO"
                indireto["PA_GERADOR"] = indireto["COD_PROD"].astype(str)
                indireto = indireto[["ID_PROD_UNID_FAT", "COD_PROD", "UNID_FAT", componente, "COEF_HR_POR_PCA", "TIPO_IMPACTO", "PA_GERADOR", "PI_IMPACTADO"]]

    partes = [x for x in [direto if 'direto' in locals() else pd.DataFrame(), indireto] if not x.empty]
    if not partes:
        return pd.DataFrame()
    cand = pd.concat(partes, ignore_index=True, sort=False)
    tipo = cand.groupby(["ID_PROD_UNID_FAT", "COD_PROD", "UNID_FAT"])["TIPO_IMPACTO"].agg(lambda s: "DIRETO" if "DIRETO" in set(s) else "INDIRETO")
    agg = cand.groupby(["ID_PROD_UNID_FAT", "COD_PROD", "UNID_FAT"], as_index=False).agg({
        componente: "first",
        "COEF_HR_POR_PCA": "sum",
        "PA_GERADOR": lambda s: "|".join(sorted({x for x in map(str, s) if x})),
        "PI_IMPACTADO": lambda s: "|".join(sorted({x for v in map(str, s) for x in v.split("|") if x})),
    })
    agg = agg.merge(tipo.rename("TIPO_IMPACTO").reset_index(), on=["ID_PROD_UNID_FAT", "COD_PROD", "UNID_FAT"], how="left")
    agg["VALOR_COMPONENTE_PCS"] = to_num(agg[componente])
    agg["HR_DISPONIVEL_CORTE"] = agg["VALOR_COMPONENTE_PCS"] * agg["COEF_HR_POR_PCA"]
    return agg[agg["HR_DISPONIVEL_CORTE"] > TOL].copy()

def aplicar_corte_batch(estado, mapa_pa_pi, candidatos, componente, corte_hr_objetivo, rodada, origem_restricao, chave_restricao, seq_criterio, criterio_texto):
    if candidatos.empty or corte_hr_objetivo <= TOL:
        return estado, pd.DataFrame()
    cand = candidatos.copy()
    total_hr = float(cand["HR_DISPONIVEL_CORTE"].sum())
    if total_hr <= TOL:
        return estado, pd.DataFrame()
    if total_hr <= corte_hr_objetivo + TOL:
        cand["CORTE_HR_PLANEJADO"] = cand["HR_DISPONIVEL_CORTE"]
    else:
        cand["CORTE_HR_PLANEJADO"] = corte_hr_objetivo * cand["HR_DISPONIVEL_CORTE"] / total_hr
    cand["CORTE_PCS"] = np.minimum(
        cand["VALOR_COMPONENTE_PCS"],
        np.divide(cand["CORTE_HR_PLANEJADO"], cand["COEF_HR_POR_PCA"], out=np.zeros(len(cand)), where=cand["COEF_HR_POR_PCA"].to_numpy(float) > TOL)
    )
    cand = cand[cand["CORTE_PCS"] > TOL].copy()
    if cand.empty:
        return estado, pd.DataFrame()

    st = estado.copy()
    st["ID_PROD_UNID_FAT"] = st["ID_PROD_UNID_FAT"].astype(str)
    ids = set(cand["ID_PROD_UNID_FAT"].astype(str))
    pos = st["ID_PROD_UNID_FAT"].isin(ids)
    antes = st.loc[pos, ["ID_PROD_UNID_FAT"] + [c for c in COLS_SNAPSHOT_MATRIZ if c in st.columns]].copy()
    antes = antes.rename(columns={c: f"ANT_{c}" for c in antes.columns if c != "ID_PROD_UNID_FAT"})
    mapa_corte = cand.set_index(cand["ID_PROD_UNID_FAT"].astype(str))["CORTE_PCS"].to_dict()
    st.loc[pos, componente] = (
        to_num(st.loc[pos, componente]) - st.loc[pos, "ID_PROD_UNID_FAT"].map(mapa_corte).fillna(0.0)
    ).clip(lower=0.0)

    nec_antes = estado.set_index(estado["ID_PROD_UNID_FAT"].astype(str))["NEC_PCS"].to_dict()
    st = recalcular_nec_estado_matriz(st)
    nec_depois = st.set_index(st["ID_PROD_UNID_FAT"].astype(str))["NEC_PCS"].to_dict()
    delta_nec = {k: max(0.0, float(nec_antes.get(k, 0.0)) - float(nec_depois.get(k, 0.0))) for k in ids}

    if mapa_pa_pi is not None and not mapa_pa_pi.empty and any(v > TOL for v in delta_nec.values()):
        rel = mapa_pa_pi[mapa_pa_pi["PA_ID"].astype(str).isin(ids)].copy()
        if not rel.empty:
            rel["DELTA_NEC_PA"] = rel["PA_ID"].astype(str).map(delta_nec).fillna(0.0)
            rel["REDUCAO_COMP_PI"] = rel["DELTA_NEC_PA"] * to_num(rel["FATOR_ESTRUTURAL"])
            red = rel.groupby("PI_ID")["REDUCAO_COMP_PI"].sum()
            st["COMP_NEC"] = (to_num(st["COMP_NEC"]) - st["ID_PROD_UNID_FAT"].astype(str).map(red).fillna(0.0)).clip(lower=0.0)
            st = recalcular_nec_estado_matriz(st)

    depois = st.loc[st["ID_PROD_UNID_FAT"].isin(ids), ["ID_PROD_UNID_FAT"] + [c for c in COLS_SNAPSHOT_MATRIZ if c in st.columns]].copy()
    depois = depois.rename(columns={c: f"DEP_{c}" for c in depois.columns if c != "ID_PROD_UNID_FAT"})
    eventos = cand.merge(antes, on="ID_PROD_UNID_FAT", how="left").merge(depois, on="ID_PROD_UNID_FAT", how="left")
    eventos["RODADA"] = rodada
    eventos["ORIGEM_RESTRICAO"] = origem_restricao
    eventos["ALOC_REC"] = str(chave_restricao) if origem_restricao == "ALOC_REC" else ""
    eventos["COD_FER_UNID"] = str(chave_restricao) if origem_restricao == "COD_FER_UNID" else ""
    eventos["CRITERIO_MATRIZ"] = f"{seq_criterio}_{criterio_texto}"
    eventos["COMPONENTE_CORTE"] = componente
    eventos["VALOR_ANTES_PCS"] = eventos[f"ANT_{componente}"]
    eventos["VALOR_DEPOIS_PCS"] = eventos[f"DEP_{componente}"]
    eventos["CORTE_PCS"] = (to_num(eventos["VALOR_ANTES_PCS"]) - to_num(eventos["VALOR_DEPOIS_PCS"])).clip(lower=0.0)
    eventos["CORTE_HR"] = eventos["CORTE_PCS"] * to_num(eventos["COEF_HR_POR_PCA"])
    eventos["MES_REF"] = st["MES_REF"].iloc[0] if "MES_REF" in st.columns and not st.empty else pd.NaT
    cols = [
        "MES_REF", "RODADA", "COD_PROD", "UNID_FAT", "ORIGEM_RESTRICAO", "ALOC_REC", "COD_FER_UNID",
        "CRITERIO_MATRIZ", "COMPONENTE_CORTE", "VALOR_ANTES_PCS", "CORTE_PCS", "VALOR_DEPOIS_PCS", "CORTE_HR",
        "TIPO_IMPACTO", "PA_GERADOR", "PI_IMPACTADO"
    ]
    snapshots = [c for c in eventos.columns if c.startswith("ANT_") or c.startswith("DEP_")]
    return st, eventos[[c for c in cols if c in eventos.columns] + snapshots].query("CORTE_PCS > @TOL").copy()

def executar_dimensao_matriz(estado, alternativas, mapa_pa_pi, dimensao, rodada_prefixo, max_passes=MATRIZ_MAX_PASSES_DIMENSAO):
    auditorias = []
    st = estado.copy()
    for tentativa in range(1, max_passes + 1):
        pend, rastro = simular_rateio_matriz(st, alternativas, dimensoes=(dimensao,))
        gaps = gaps_por_restricao(pend, rastro, dimensao)
        if gaps.empty or gaps["CORTE_HR_NECESSARIO"].sum() <= TOL:
            return st, pd.concat(auditorias, ignore_index=True, sort=False) if auditorias else pd.DataFrame(), True
        houve_corte = False
        for chave, grp in gaps.groupby(dimensao, sort=False):
            corte_hr = float(grp["CORTE_HR_NECESSARIO"].sum())
            participantes = participantes_restricao(rastro, dimensao, chave)
            for seq, mesma_reg, componente in MATRIZ_CRITERIOS:
                if corte_hr <= TOL:
                    break
                cand = candidatos_criterio(st, mapa_pa_pi, participantes, mesma_reg, componente)
                if cand.empty:
                    continue
                st_novo, ev = aplicar_corte_batch(
                    st, mapa_pa_pi, cand, componente, corte_hr,
                    f"{rodada_prefixo}_P{tentativa}", dimensao, chave, seq, f"{mesma_reg}|{componente}"
                )
                if not ev.empty:
                    houve_corte = True
                    auditorias.append(ev)
                    st = st_novo
                    pend2, rastro2 = simular_rateio_matriz(st, alternativas, dimensoes=(dimensao,))
                    gaps2 = gaps_por_restricao(pend2, rastro2, dimensao)
                    corte_hr = float(gaps2.loc[gaps2[dimensao].astype(str).eq(str(chave)), "CORTE_HR_NECESSARIO"].sum()) if not gaps2.empty else 0.0
                    rastro = rastro2
                    participantes = participantes_restricao(rastro, dimensao, chave)
        if not houve_corte:
            return st, pd.concat(auditorias, ignore_index=True, sort=False) if auditorias else pd.DataFrame(), True
    return st, pd.concat(auditorias, ignore_index=True, sort=False) if auditorias else pd.DataFrame(), False

def existe_corte_matriz_disponivel(estado, alternativas, mapa_pa_pi):
    pend, rastro = simular_rateio_matriz(estado, alternativas, dimensoes=("ALOC_REC", "COD_FER_UNID"))
    if pend.empty or float(pend.clip(lower=0.0).sum()) <= TOL:
        return False, float(pend.clip(lower=0.0).sum()) if not pend.empty else 0.0
    for dimensao in ["ALOC_REC", "COD_FER_UNID"]:
        gaps = gaps_por_restricao(pend, rastro, dimensao)
        for chave, _ in gaps.groupby(dimensao, sort=False):
            participantes = participantes_restricao(rastro, dimensao, chave)
            for _, mesma_reg, componente in MATRIZ_CRITERIOS:
                cand = candidatos_criterio(estado, mapa_pa_pi, participantes, mesma_reg, componente)
                if not cand.empty and float(cand["HR_DISPONIVEL_CORTE"].sum()) > TOL:
                    return True, float(pend.clip(lower=0.0).sum())
    return False, float(pend.clip(lower=0.0).sum())

def executar_matriz_cortes(bd_mes, estrutura_explodida):
    bd_rotas, estado = preparar_estado_matriz(bd_mes)
    alternativas = preparar_alternativas_matriz(bd_rotas)
    mapa_pa_pi = construir_mapa_impacto_pa_pi(estrutura_explodida, estado)
    auditorias = []

    estado, aud, ok_rec = executar_dimensao_matriz(estado, alternativas, mapa_pa_pi, "ALOC_REC", "1_ALOC_REC")
    if not aud.empty:
        auditorias.append(aud)
    estado, aud, ok_fer = executar_dimensao_matriz(estado, alternativas, mapa_pa_pi, "COD_FER_UNID", "2_COD_FER_UNID")
    if not aud.empty:
        auditorias.append(aud)

    reconciliacoes = 0
    for rec in range(1, MATRIZ_MAX_RECONCILIACOES + 1):
        existe, _ = existe_corte_matriz_disponivel(estado, alternativas, mapa_pa_pi)
        if not existe:
            break
        houve = False
        for dimensao in ["ALOC_REC", "COD_FER_UNID"]:
            novo, aud, _ = executar_dimensao_matriz(estado, alternativas, mapa_pa_pi, dimensao, f"R{rec}_{dimensao}", max_passes=2)
            if not aud.empty:
                houve = True
                auditorias.append(aud)
                estado = novo
        reconciliacoes = rec
        if not houve:
            break

    corte_disponivel, gap_residual = existe_corte_matriz_disponivel(estado, alternativas, mapa_pa_pi)
    convergiu = bool(ok_rec and ok_fer and not corte_disponivel)
    if not convergiu:
        raise RuntimeError(
            f"MATRIZ_NAO_CONVERGIU: reconciliacoes={reconciliacoes}; gap_residual_pcs={gap_residual:.6f}; ainda_existe_corte_valido={corte_disponivel}"
        )

    ch = "ID_PROD_UNID_FAT"
    cols_estado = [
        ch, "C_ARR", "C_AT", "PREV_VEN", "PREV_PROX", "EST_SEG", "EST_IN", "EST_TRANS", "COMP_NEC", "NEC_PCS",
        "ORI_C_ARR", "ORI_C_AT", "ORI_PREV_VEN", "ORI_PREV_PROX", "ORI_EST_SEG", "ORI_NEC"
    ]
    mapa_estado = estado[[c for c in cols_estado if c in estado.columns]].copy()
    bd_final = bd_rotas.drop(columns=[c for c in cols_estado if c != ch and c in bd_rotas.columns], errors="ignore").merge(
        mapa_estado, on=ch, how="left", validate="many_to_one"
    )
    for curto, origem in MAP_COMPONENTE_LTP.items():
        bd_final[origem] = to_num(bd_final[curto])
    bd_final["NEC_COMP_PCS"] = to_num(bd_final["COMP_NEC"])
    bd_final["NEC_EXPLODIDA_PCS"] = to_num(bd_final["COMP_NEC"])
    bd_final["NEC_EXPLODIDA_BRUTA_PCS"] = to_num(bd_final["COMP_NEC"])
    bd_final["NEC_PCS_CONSOLIDADA"] = to_num(bd_final["NEC_PCS"])
    bd_final["NEC_DIRETA_PCS"] = (to_num(bd_final["NEC_PCS"]) - to_num(bd_final["COMP_NEC"])).clip(lower=0.0)

    chave_item = [c for c in ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "UNID_PROD", "UNID_FAT"] if c in bd_final.columns]
    agg = {c: "max" for c in ["NEC_DIRETA_PCS", "NEC_COMP_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_PCS_CONSOLIDADA"] if c in bd_final.columns}
    for c in ["DESC_PROD", "TIPO_PROD"]:
        if c in bd_final.columns:
            agg[c] = "first"
    nec_consolidada = bd_final.groupby(chave_item, as_index=False).agg(agg)
    nec_consolidada["TEM_ROTEIRO_NO_SOLVER"] = nec_consolidada["ID_PROD_UNID_FAT"].notna()

    auditoria = pd.concat(auditorias, ignore_index=True, sort=False) if auditorias else pd.DataFrame(columns=[
        "MES_REF", "RODADA", "COD_PROD", "UNID_FAT", "ORIGEM_RESTRICAO", "ALOC_REC", "COD_FER_UNID",
        "CRITERIO_MATRIZ", "COMPONENTE_CORTE", "VALOR_ANTES_PCS", "CORTE_PCS", "VALOR_DEPOIS_PCS", "CORTE_HR",
        "TIPO_IMPACTO", "PA_GERADOR", "PI_IMPACTADO"
    ])
    validacao = {
        "MATRIZ_CONVERGIU": convergiu,
        "MATRIZ_GAP_RESIDUAL_PCS": gap_residual,
        "MATRIZ_RECONCILIACOES": reconciliacoes,
    }
    return bd_final, nec_consolidada, auditoria, validacao

print("[04/10] OK - Matriz de Cortes carregada com 9 criterios fixos")


In [ ]:
# ============================================================
# ETAPA 6 - SOLVER DE CAPACIDADE
# OBJETIVO: alocar a NEC pos-Matriz respeitando dupla restricao global e prioridades de rota.
# ENTRADAS: NEC final da Matriz e alternativas validas.
# PODE ALTERAR: quantidade/hora alocada por rota.
# DEVE PRESERVAR: PRIOR_MATPAR, PRIOR_ROT, ALOC_REC, COD_FER_UNID e HOR_CAP sem fallback.
# SAIDAS: plano continuo por rota e rastro interno.
# VALIDACAO ESPERADA: nenhuma capacidade ultrapassada.
# ============================================================

def executar_rateio_proporcional_dupla_restricao(bd_demanda, bd_alternativas_validas, bd_cap_recursos, bd_cap_ferramentas):
    alternativas = bd_alternativas_validas.copy()
    cap_restante_rec = bd_cap_recursos.set_index("ALOC_REC")["HOR_REC"].to_dict()
    cap_restante_fer = bd_cap_ferramentas.set_index("COD_FER_UNID")["HOR_FER"].to_dict()
    pendente = bd_demanda.set_index("ID_PROD_UNID_FAT")["NEC_PCS"].to_dict()
    producao, rastro = [], []

    niveis_prioridade = list(alternativas[["PRIOR_MATPAR", "PRIOR_ROT"]].drop_duplicates().sort_values(["PRIOR_MATPAR", "PRIOR_ROT"]).itertuples(index=False, name=None))
    for rodada, (prior_matpar, prior_rot) in enumerate(niveis_prioridade, start=1):
        nivel = alternativas[alternativas["PRIOR_MATPAR"].eq(prior_matpar) & alternativas["PRIOR_ROT"].eq(prior_rot)].copy()
        nivel["RODADA"] = rodada
        nivel["QTD_PENDENTE_ANTES"] = nivel["ID_PROD_UNID_FAT"].map(pendente).fillna(0)
        nivel = nivel[nivel["QTD_PENDENTE_ANTES"] > TOL].copy()
        if nivel.empty:
            continue

        qtd_alt_mesmo_nivel = nivel.groupby("ID_PROD_UNID_FAT")["ALOC_REC"].transform("count")
        nivel["QTD_ALTERNATIVAS_MESMO_NIVEL"] = qtd_alt_mesmo_nivel
        nivel["QTD_SOLICITADA"] = nivel["QTD_PENDENTE_ANTES"] / qtd_alt_mesmo_nivel
        nivel["HR_SOLICITADA"] = nivel["QTD_SOLICITADA"] / nivel["PCS_HORA"]

        nivel["CAP_RESTANTE_ALOC_REC_ANTES"] = nivel["ALOC_REC"].map(cap_restante_rec).fillna(0)
        nivel["CAP_RESTANTE_COD_FER_ANTES"] = nivel["COD_FER_UNID"].map(cap_restante_fer).fillna(0)

        demanda_rec = nivel.groupby("ALOC_REC")["HR_SOLICITADA"].transform("sum")
        demanda_fer = nivel.groupby("COD_FER_UNID")["HR_SOLICITADA"].transform("sum")
        nivel["HR_DEMANDA_TOTAL_ALOC_REC"] = demanda_rec
        nivel["HR_DEMANDA_TOTAL_COD_FER"] = demanda_fer
        nivel["FATOR_RATEIO_ALOC_REC"] = np.where(demanda_rec > TOL, np.minimum(1.0, nivel["CAP_RESTANTE_ALOC_REC_ANTES"] / demanda_rec), 0.0)
        nivel["FATOR_RATEIO_COD_FER"] = np.where(demanda_fer > TOL, np.minimum(1.0, nivel["CAP_RESTANTE_COD_FER_ANTES"] / demanda_fer), 0.0)
        nivel["FATOR_RATEIO"] = nivel[["FATOR_RATEIO_ALOC_REC", "FATOR_RATEIO_COD_FER"]].min(axis=1).clip(lower=0, upper=1)
        nivel["CAP_DISPONIVEL_LINHA"] = nivel[["CAP_RESTANTE_ALOC_REC_ANTES", "CAP_RESTANTE_COD_FER_ANTES"]].min(axis=1)

        cond_rec = nivel["FATOR_RATEIO_ALOC_REC"] < nivel["FATOR_RATEIO_COD_FER"] - TOL
        cond_fer = nivel["FATOR_RATEIO_COD_FER"] < nivel["FATOR_RATEIO_ALOC_REC"] - TOL
        cond_ambos = (~cond_rec) & (~cond_fer) & (nivel["FATOR_RATEIO"] < 1 - TOL)
        nivel["MOTIVO_LIMITANTE"] = np.select([cond_rec, cond_fer, cond_ambos], ["ALOC_REC", "COD_FER_UNID", "ALOC_REC_E_COD_FER"], default="SEM_LIMITACAO")

        # O rateio proporcional permanece contínuo nesta etapa e calcula somente a distribuição factível
        # pela dupla restrição de recurso + ferramenta. LIMIT_PCS/LOTE_MIN/QTD_EMB entram somente
        # no fechamento industrial posterior. Depois desse fechamento, a capacidade nao volta a cortar o plano.
        nivel["HR_PRODUZIR_SOLVER"] = nivel["HR_SOLICITADA"] * nivel["FATOR_RATEIO"]
        nivel["QTD_PRODUZIR_SOLVER"] = (nivel["HR_PRODUZIR_SOLVER"] * nivel["PCS_HORA"]).clip(lower=0.0)
        nivel["QTD_PENDENTE_DEPOIS"] = (nivel["QTD_PENDENTE_ANTES"] - nivel.groupby("ID_PROD_UNID_FAT")["QTD_PRODUZIR_SOLVER"].transform("sum")).clip(lower=0)

        nivel["STATUS_DECISAO"] = np.select(
            [nivel["CAP_RESTANTE_ALOC_REC_ANTES"] <= TOL, nivel["CAP_RESTANTE_COD_FER_ANTES"] <= TOL, nivel["HR_PRODUZIR_SOLVER"] <= TOL, nivel["FATOR_RATEIO"] >= 1 - TOL, nivel["FATOR_RATEIO"] < 1 - TOL],
            ["NAO_ALOCADO_CAP_RECURSO_ZERO", "NAO_ALOCADO_CAP_FERRAMENTA_ZERO", "NAO_ALOCADO_RATEIO_ZERO", "ALOCADO_TOTAL_RODADA", "ALOCADO_PARCIAL_RATEIO"],
            default="VERIFICAR"
        )
        nivel["MOTIVO_CORTE"] = np.where(nivel["FATOR_RATEIO"] >= 1 - TOL, "SEM_CORTE", "CORTE_POR_" + nivel["MOTIVO_LIMITANTE"].astype(str))

        for aloc_rec, consumo in nivel.groupby("ALOC_REC")["HR_PRODUZIR_SOLVER"].sum().items():
            cap_restante_rec[aloc_rec] = max(0.0, cap_restante_rec.get(aloc_rec, 0.0) - float(consumo))
        for cod_fer, consumo in nivel.groupby("COD_FER_UNID")["HR_PRODUZIR_SOLVER"].sum().items():
            cap_restante_fer[cod_fer] = max(0.0, cap_restante_fer.get(cod_fer, 0.0) - float(consumo))
        for item, qtd_prod in nivel.groupby("ID_PROD_UNID_FAT")["QTD_PRODUZIR_SOLVER"].sum().items():
            pendente[item] = max(0.0, pendente.get(item, 0.0) - float(qtd_prod))

        producao.append(nivel[nivel["QTD_PRODUZIR_SOLVER"] > TOL].copy())
        rastro.append(nivel.copy())

    bd_producao = pd.concat(producao, ignore_index=True) if producao else pd.DataFrame()
    bd_rastro = pd.concat(rastro, ignore_index=True) if rastro else pd.DataFrame()
    return bd_producao, bd_rastro, pendente, cap_restante_rec, cap_restante_fer


def preparar_solver_capacidade(bd_mes):
    meta = [
        "MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD",
        "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG", "NEC_PCS_ORIGINAL", "NEC_ESTRUTURA_PAI_PCS",
        "NEC_DIRETA_BRUTA_PCS", "NEC_DIRETA_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS",
        "NEC_EXPLODIDA_PCS", "NEC_COMP_PCS", "NEC_PCS_CONSOLIDADA", "LOTE_MIN", "QTD_EMB", "CURVA_ABC", "CURVA_123", "MESMA_REG"
    ]
    demanda = (
        bd_mes.sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])
        .groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False)
        .agg({**{c: "first" for c in meta if c not in ["MES_REF", "ID_PROD_UNID_FAT"] and c in bd_mes.columns}, "NEC_PCS": "max"})
    )
    demanda = demanda[demanda["NEC_PCS"] > TOL].copy()

    cols_alt = [
        "MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID",
        "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD",
        "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG", "ID_RECURSO", "ID_FERRAMENTA", "LOTE_MIN", "QTD_EMB", "CURVA_ABC", "CURVA_123", "MESMA_REG"
    ]
    alt = bd_mes[[c for c in cols_alt if c in bd_mes.columns]].copy()
    alt = alt[alt["ID_PROD_UNID_FAT"].isin(demanda["ID_PROD_UNID_FAT"])].copy()
    alt = alt.drop_duplicates(["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID"]).reset_index(drop=True)
    for c in ["PRIOR_MATPAR", "PRIOR_ROT", "PCS_HORA", "HOR_REC", "HOR_FER"]:
        alt[c] = to_num(alt[c])
    alt["HOR_CAP"] = alt[["HOR_REC", "HOR_FER"]].min(axis=1)
    alt["ALTERNATIVA_VALIDA"] = (
        alt["ALOC_REC"].notna() & alt["COD_FER_UNID"].notna()
        & (alt["PCS_HORA"] > TOL) & (alt["HOR_REC"] > TOL) & (alt["HOR_FER"] > TOL) & (alt["HOR_CAP"] > TOL)
    )
    conds = [
        alt["ALOC_REC"].isna(), alt["COD_FER_UNID"].isna(), alt["PCS_HORA"] <= TOL,
        alt["HOR_REC"] <= TOL, alt["HOR_FER"] <= TOL, alt["HOR_CAP"] <= TOL,
    ]
    escolhas = ["ALOC_REC_AUSENTE", "COD_FER_UNID_AUSENTE", "PCS_HORA_ZERO_OU_INVALIDO", "HOR_REC_ZERO_OU_NEGATIVO", "HOR_FER_ZERO_OU_NEGATIVO", "HOR_CAP_ZERO_OU_NEGATIVO"]
    alt["MOTIVO_INVALIDA"] = np.select(conds, escolhas, default="ALTERNATIVA_VALIDA")
    validas = alt[alt["ALTERNATIVA_VALIDA"]].copy()
    cap_rec = alt.dropna(subset=["ALOC_REC"]).groupby("ALOC_REC", as_index=False).agg(HOR_REC=("HOR_REC", "max"))
    cap_fer = alt.dropna(subset=["COD_FER_UNID"]).groupby("COD_FER_UNID", as_index=False).agg(HOR_FER=("HOR_FER", "max"))
    return demanda, alt, validas, cap_rec, cap_fer

def executar_solver_capacidade(bd_mes):
    demanda, alternativas, validas, cap_rec, cap_fer = preparar_solver_capacidade(bd_mes)
    plano, rastro, pendente, saldo_rec, saldo_fer = executar_rateio_proporcional_dupla_restricao(
        demanda, validas, cap_rec, cap_fer
    )
    return {
        "demanda": demanda, "alternativas": alternativas, "validas": validas,
        "cap_rec": cap_rec, "cap_fer": cap_fer, "plano": plano, "rastro": rastro,
        "pendente": pendente, "saldo_rec": saldo_rec, "saldo_fer": saldo_fer,
    }

print("[05/10] OK - Solver de capacidade preparado")


In [ ]:
# ============================================================
# ETAPA 7 - FECHAMENTO OPERACIONAL
# OBJETIVO: transformar o plano continuo em pecas inteiras validas por LOTE_MIN/QTD_EMB sem aumentar uso de capacidade.
# ENTRADAS: alocacao continua do Solver e NEC final.
# PODE ALTERAR: QTD_PRODUZIR_SOLVER e horas derivadas somente para baixo.
# DEVE PRESERVAR: capacidade, NEC, lote minimo, multiplo e prioridade das rotas.
# SAIDAS: 03_PLANO_PRODUCAO completo, incluindo itens zerados.
# VALIDACAO ESPERADA: zero fracao, zero multiplo invalido e zero estouro criado pelo fechamento.
# ============================================================

def fechar_plano_operacional(solver):
    """Fecha o plano em blocos operacionais validos e reaproveita capacidade residual.

    Fases:
    1. materializa todas as alternativas validas com a quantidade continua alocada pelo Solver;
    2. reduz a quantidade para pecas inteiras e fecha o total do item em LOTE_MIN/QTD_EMB;
    3. recalcula capacidade residual real de ALOC_REC e COD_FER_UNID;
    4. usa a sobra para adicionar o maior numero de blocos validos que couber, sem ultrapassar NEC.

    Nenhuma etapa arredonda quantidade para cima sem antes provar capacidade disponivel.
    """
    alternativas = solver["validas"].copy()
    demanda = solver["demanda"].copy()
    continuo = solver["plano"].copy()
    if alternativas.empty or demanda.empty:
        return pd.DataFrame()

    chave_rota = ["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID"]
    if continuo.empty:
        prod_cont = pd.DataFrame(columns=chave_rota + ["QTD_CONTINUA"])
    else:
        prod_cont = (
            continuo.groupby(chave_rota, as_index=False)
            .agg(QTD_CONTINUA=("QTD_PRODUZIR_SOLVER", "sum"))
        )

    p = alternativas.merge(prod_cont, on=chave_rota, how="left")
    p["QTD_CONTINUA"] = to_num(p["QTD_CONTINUA"])
    for c in ["PCS_HORA", "LOTE_MIN", "QTD_EMB", "PRIOR_MATPAR", "PRIOR_ROT"]:
        p[c] = to_num(p[c])

    item_meta = demanda[[c for c in ["MES_REF", "ID_PROD_UNID_FAT", "NEC_PCS", "LOTE_MIN", "QTD_EMB"] if c in demanda.columns]].copy()
    for c in ["NEC_PCS", "LOTE_MIN", "QTD_EMB"]:
        item_meta[c] = to_num(item_meta[c])
    item_meta = item_meta.drop_duplicates(["MES_REF", "ID_PROD_UNID_FAT"])
    p = p.drop(columns=[c for c in ["NEC_PCS", "LOTE_MIN", "QTD_EMB"] if c in p.columns], errors="ignore").merge(
        item_meta, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left", validate="many_to_one"
    )

    # Fechamento inicial: nunca excede a quantidade continua da rota.
    p["CAP_PCS_INTEIRA_ROTA"] = np.floor(p["QTD_CONTINUA"].clip(lower=0.0) + TOL)
    chave_item = ["MES_REF", "ID_PROD_UNID_FAT"]
    item = p.groupby(chave_item, as_index=False).agg(
        CAP_PCS=("CAP_PCS_INTEIRA_ROTA", "sum"), NEC_PCS=("NEC_PCS", "max"),
        LOTE_MIN=("LOTE_MIN", "max"), QTD_EMB=("QTD_EMB", "max")
    )
    item["MIN_POSITIVO"] = np.where(
        item["QTD_EMB"] > TOL,
        np.ceil(np.maximum(item["LOTE_MIN"], 0.0) / item["QTD_EMB"].replace(0, np.nan)).fillna(0.0) * item["QTD_EMB"],
        np.ceil(np.maximum(item["LOTE_MIN"], 0.0)),
    )
    item["QTD_FINAL_ITEM"] = np.where(
        item["QTD_EMB"] > TOL,
        np.floor((item["CAP_PCS"] + TOL) / item["QTD_EMB"]) * item["QTD_EMB"],
        np.floor(item["CAP_PCS"] + TOL),
    )
    item["QTD_FINAL_ITEM"] = np.minimum(item["QTD_FINAL_ITEM"], np.floor(item["NEC_PCS"] + TOL))
    item.loc[(item["QTD_FINAL_ITEM"] > TOL) & (item["QTD_FINAL_ITEM"] + TOL < item["MIN_POSITIVO"]), "QTD_FINAL_ITEM"] = 0.0
    p = p.merge(item[chave_item + ["QTD_FINAL_ITEM", "MIN_POSITIVO"]], on=chave_item, how="left", validate="many_to_one")

    ordem = ["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID"]
    p = p.sort_values(ordem).copy()
    p["CAP_ACUM"] = p.groupby(chave_item)["CAP_PCS_INTEIRA_ROTA"].cumsum()
    p["CAP_ACUM_ANTES"] = p["CAP_ACUM"] - p["CAP_PCS_INTEIRA_ROTA"]
    p["QTD_PRODUZIR_SOLVER"] = np.minimum(
        p["CAP_PCS_INTEIRA_ROTA"],
        np.maximum(p["QTD_FINAL_ITEM"] - p["CAP_ACUM_ANTES"], 0.0),
    )
    p["QTD_PRODUZIR_SOLVER"] = np.floor(p["QTD_PRODUZIR_SOLVER"] + TOL)

    # Capacidades residuais reais depois do fechamento inicial.
    p["HR_PRODUZIR_SOLVER"] = np.divide(
        p["QTD_PRODUZIR_SOLVER"].to_numpy(float), p["PCS_HORA"].to_numpy(float),
        out=np.zeros(len(p)), where=p["PCS_HORA"].to_numpy(float) > TOL,
    )
    uso_rec = p.groupby("ALOC_REC")["HR_PRODUZIR_SOLVER"].sum().to_dict()
    uso_fer = p.groupby("COD_FER_UNID")["HR_PRODUZIR_SOLVER"].sum().to_dict()
    cap_rec = solver["cap_rec"].set_index("ALOC_REC")["HOR_REC"].to_dict()
    cap_fer = solver["cap_fer"].set_index("COD_FER_UNID")["HOR_FER"].to_dict()
    saldo_rec = {k: max(0.0, float(v) - float(uso_rec.get(k, 0.0))) for k, v in cap_rec.items()}
    saldo_fer = {k: max(0.0, float(v) - float(uso_fer.get(k, 0.0))) for k, v in cap_fer.items()}

    total_item = p.groupby(chave_item)["QTD_PRODUZIR_SOLVER"].sum().to_dict()
    nec_item = item.set_index(chave_item)["NEC_PCS"].to_dict()
    lote_item = item.set_index(chave_item)["LOTE_MIN"].to_dict()
    emb_item = item.set_index(chave_item)["QTD_EMB"].to_dict()
    min_item = item.set_index(chave_item)["MIN_POSITIVO"].to_dict()

    # Uma passagem por alternativa, em prioridade de roteiro. Cada linha recebe de uma vez
    # o maior numero de blocos que cabe; portanto nao existe loop por peca/bloco unitario.
    candidatos = p.copy()
    candidatos["BLOCO_POSITIVO"] = np.where(
        candidatos["QTD_EMB"] > TOL, candidatos["QTD_EMB"], 1.0
    )
    candidatos["HR_BLOCO"] = np.divide(
        candidatos["BLOCO_POSITIVO"].to_numpy(float), candidatos["PCS_HORA"].to_numpy(float),
        out=np.full(len(candidatos), np.inf), where=candidatos["PCS_HORA"].to_numpy(float) > TOL,
    )
    candidatos = candidatos.sort_values(
        ["PRIOR_MATPAR", "PRIOR_ROT", "HR_BLOCO", "ID_PROD_UNID_FAT", "ALOC_REC", "COD_FER_UNID"]
    )

    idx_por_chave = {tuple(k): list(v) for k, v in p.groupby(chave_rota, sort=False).groups.items()}
    for r in candidatos.itertuples(index=False):
        item_key = (r.MES_REF, r.ID_PROD_UNID_FAT)
        atual = float(total_item.get(item_key, 0.0))
        nec = float(nec_item.get(item_key, 0.0))
        gap = max(0.0, nec - atual)
        if gap <= TOL or float(r.PCS_HORA) <= TOL:
            continue

        emb = float(max(0.0, emb_item.get(item_key, 0.0)))
        lote = float(max(0.0, lote_item.get(item_key, 0.0)))
        minimo = float(max(0.0, min_item.get(item_key, 0.0)))
        bloco = emb if emb > TOL else 1.0
        if atual <= TOL and minimo > TOL:
            bloco_inicial = minimo
        else:
            bloco_inicial = bloco
        if bloco_inicial <= TOL or gap + TOL < bloco_inicial:
            continue

        rec = r.ALOC_REC
        fer = r.COD_FER_UNID
        hr_disp = min(float(saldo_rec.get(rec, 0.0)), float(saldo_fer.get(fer, 0.0)))
        if hr_disp <= TOL:
            continue
        qtd_cap = np.floor(hr_disp * float(r.PCS_HORA) + TOL)
        if qtd_cap + TOL < bloco_inicial:
            continue

        if atual <= TOL and minimo > TOL:
            qtd_add = bloco_inicial
            restante_gap = max(0.0, gap - qtd_add)
            restante_cap = max(0.0, qtd_cap - qtd_add)
            if restante_gap > TOL and restante_cap > TOL:
                extras = min(np.floor((restante_gap + TOL) / bloco), np.floor((restante_cap + TOL) / bloco))
                qtd_add += max(0.0, extras) * bloco
        else:
            blocos = min(np.floor((gap + TOL) / bloco), np.floor((qtd_cap + TOL) / bloco))
            qtd_add = max(0.0, blocos) * bloco

        qtd_add = np.floor(qtd_add + TOL)
        if qtd_add <= TOL:
            continue
        hr_add = qtd_add / float(r.PCS_HORA)
        if hr_add > saldo_rec.get(rec, 0.0) + TOL or hr_add > saldo_fer.get(fer, 0.0) + TOL:
            continue

        route_key = (r.MES_REF, r.ID_PROD_UNID_FAT, r.PRIOR_MATPAR, r.PRIOR_ROT, r.ALOC_REC, r.COD_FER_UNID)
        idxs = idx_por_chave.get(route_key, [])
        if not idxs:
            continue
        idx = idxs[0]
        p.at[idx, "QTD_PRODUZIR_SOLVER"] = float(p.at[idx, "QTD_PRODUZIR_SOLVER"]) + qtd_add
        p.at[idx, "HR_PRODUZIR_SOLVER"] = float(p.at[idx, "QTD_PRODUZIR_SOLVER"]) / float(r.PCS_HORA)
        total_item[item_key] = atual + qtd_add
        saldo_rec[rec] = max(0.0, float(saldo_rec.get(rec, 0.0)) - hr_add)
        saldo_fer[fer] = max(0.0, float(saldo_fer.get(fer, 0.0)) - hr_add)

    if "PESO_PROD_KG" in p.columns:
        p["VOL_PRODUZIR_SOLVER"] = p["QTD_PRODUZIR_SOLVER"] * to_num(p["PESO_PROD_KG"])
    return p.drop(columns=[
        "QTD_CONTINUA", "CAP_PCS_INTEIRA_ROTA", "QTD_FINAL_ITEM", "MIN_POSITIVO", "CAP_ACUM", "CAP_ACUM_ANTES"
    ], errors="ignore")

def classificar_status_zero(base):
    out = base.copy()
    for c in ["NEC_PCS", "NEC_COMP_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "NEC_EXPLODIDA_PCS", "LTP_EST_INI_PCS", "LTP_EST_TRANS_PCS", "ORI_TOT_PCS", "TRIANG_TOT_PCS"]:
        if c not in out.columns:
            out[c] = 0.0
        out[c] = to_num(out[c])
    demanda_bruta = sum(to_num(out[c]) if c in out.columns else 0 for c in ["LTP_CART_ARR_MES_ANT", "LTP_CART_MES_ATUAL", "LTP_SALDO_PREV_PCS", "LTP_EST_SEG_PCS"])
    if "LTP_SALDO_PREV_PROX_MES_PCS" in out.columns:
        demanda_bruta = demanda_bruta + np.where(out["MESMA_REG"].astype(str).eq("NAO"), to_num(out["LTP_SALDO_PREV_PROX_MES_PCS"]), 0.0)
    cobertura = out["LTP_EST_INI_PCS"] + out["LTP_EST_TRANS_PCS"] + out["ORI_TOT_PCS"] + out["TRIANG_TOT_PCS"]
    out["MOTIVO_STATUS"] = np.select(
        [
            (out["NEC_PCS"] <= TOL) & (demanda_bruta > TOL) & (cobertura + TOL >= demanda_bruta),
            (out["NEC_PCS"] <= TOL) & (out["NEC_EXPLODIDA_BRUTA_PCS"] > TOL) & (out["NEC_EXPLODIDA_PCS"] <= TOL),
            out["NEC_PCS"] <= TOL,
        ],
        ["DEMANDA_DIRETA_COBERTA_POR_ESTOQUE", "DEMANDA_ESTRUTURAL_COBERTA_POR_ESTOQUE", "ITEM_SEM_DEMANDA_NO_MES"],
        default="NAO_ALOCADO_SEM_ALTERNATIVA_VALIDA",
    )
    return out

def montar_plano_producao(bd_mes, solver):
    plano = fechar_plano_operacional(solver)
    if plano.empty:
        plano = pd.DataFrame(columns=list(solver["alternativas"].columns) + ["QTD_PRODUZIR_SOLVER", "HR_PRODUZIR_SOLVER"])

    item_cols = [
        "MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD",
        "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG", "QTD_EMB", "LOTE_MIN", "CURVA_ABC", "CURVA_123", "MESMA_REG",
        "ORI_C_ARR", "C_ARR", "ORI_C_AT", "C_AT", "ORI_PREV_VEN", "PREV_VEN", "ORI_PREV_PROX", "PREV_PROX",
        "ORI_EST_SEG", "EST_SEG", "ORI_NEC", "NEC_PCS", "EST_IN", "EST_TRANS", "COMP_NEC", "ORI_TOT_PCS", "TRIANG_TOT_PCS",
        "NEC_DIRETA_PCS", "NEC_COMP_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_PCS_CONSOLIDADA",
    ]
    ordem = [c for c in ["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID"] if c in bd_mes.columns]
    base_item = bd_mes.sort_values(ordem).drop_duplicates(["MES_REF", "ID_PROD_UNID_FAT"], keep="first")
    base_item = base_item[[c for c in item_cols if c in base_item.columns]].copy()
    base_item = classificar_status_zero(base_item)

    meta_merge = [c for c in item_cols if c not in ["MES_REF", "ID_PROD_UNID_FAT"] and c in base_item.columns]
    plano = plano.drop(columns=[c for c in meta_merge if c in plano.columns], errors="ignore").merge(
        base_item[["MES_REF", "ID_PROD_UNID_FAT"] + meta_merge + ["MOTIVO_STATUS"]],
        on=["MES_REF", "ID_PROD_UNID_FAT"], how="left", validate="many_to_one"
    )
    plano["QTD_PRODUZIR_SOLVER"] = to_num(plano.get("QTD_PRODUZIR_SOLVER", 0.0))
    plano.loc[plano["QTD_PRODUZIR_SOLVER"] > TOL, "MOTIVO_STATUS"] = "PRODUCAO_ALOCADA_PELO_SOLVER"

    produzidos = set(zip(plano["MES_REF"], plano["ID_PROD_UNID_FAT"].astype(str))) if not plano.empty else set()
    faltantes = base_item[[ (m, str(i)) not in produzidos for m, i in zip(base_item["MES_REF"], base_item["ID_PROD_UNID_FAT"]) ]].copy()
    if not faltantes.empty:
        rep_cols = [
            "MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP"
        ]
        rep = bd_mes.sort_values(ordem).drop_duplicates(["MES_REF", "ID_PROD_UNID_FAT"], keep="first")[[c for c in rep_cols if c in bd_mes.columns]].copy()
        zero = faltantes.merge(rep, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left", validate="one_to_one")
        zero["QTD_PRODUZIR_SOLVER"] = 0.0
        zero["HR_PRODUZIR_SOLVER"] = 0.0
        zero["VOL_PRODUZIR_SOLVER"] = 0.0
        zero["STATUS_DECISAO"] = "NAO_PLANEJADO"
        zero["MOTIVO_LIMITANTE"] = "NAO_APLICAVEL"
        zero["MOTIVO_CORTE"] = "SEM_CORTE"
        plano = pd.concat([plano, zero], ignore_index=True, sort=False)

    prod_item = plano.groupby(["MES_REF", "ID_PROD_UNID_FAT"])["QTD_PRODUZIR_SOLVER"].transform("sum")
    plano["QTD_GAP_FINAL"] = (to_num(plano["NEC_PCS"]) - prod_item).clip(lower=0.0)
    plano["VOL_PRODUZIR_SOLVER"] = to_num(plano["QTD_PRODUZIR_SOLVER"]) * to_num(plano.get("PESO_PROD_KG", 0.0))

    prefixo = [
        "MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD",
        "QTD_EMB", "LOTE_MIN", "CURVA_ABC", "CURVA_123", "MESMA_REG",
        "ORI_C_ARR", "C_ARR", "ORI_C_AT", "C_AT", "ORI_PREV_VEN", "PREV_VEN", "ORI_PREV_PROX", "PREV_PROX",
        "ORI_EST_SEG", "EST_SEG", "ORI_NEC", "NEC_PCS", "EST_IN", "EST_TRANS", "COMP_NEC", "ORI_TOT_PCS", "TRIANG_TOT_PCS",
    ]
    return plano[[c for c in prefixo if c in plano.columns] + [c for c in plano.columns if c not in prefixo]].sort_values(
        [c for c in ["MES_REF", "COD_PROD", "UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"] if c in plano.columns]
    ).reset_index(drop=True)

print("[06/10] OK - Fechamento operacional vetorizado preparado")


In [ ]:
# ============================================================
# ETAPA 8 - VALIDACAO DO MES
# OBJETIVO: provar cada regra critica com resultado PASSOU/FALHOU.
# ENTRADAS: plano final, auditoria da Matriz, capacidades e NEC final.
# PODE ALTERAR: somente tabelas de validacao/diagnostico.
# DEVE PRESERVAR: todos os resultados de negocio.
# SAIDAS: validacao_periodo e excecao detalhada quando houver falha critica.
# VALIDACAO ESPERADA: nenhuma mensagem generica substitui a prova numerica.
# ============================================================

def validar_periodo(plano, auditoria_matriz, solver, validacao_matriz, mes_ref):
    registros = []
    falhas = []
    def add(regra, ok, detalhe=""):
        registros.append({"MES_REF": pd.Timestamp(mes_ref), "REGRA": regra, "STATUS": "PASSOU" if ok else "FALHOU", "DETALHE": detalhe})
        if not ok:
            falhas.append(f"{regra}: {detalhe}")

    positivos = plano[to_num(plano["QTD_PRODUZIR_SOLVER"]) > TOL].copy()
    qtd = to_num(plano["QTD_PRODUZIR_SOLVER"])
    add("SEM_FRACAO_PECA", bool(np.all(np.isclose(qtd, np.floor(qtd + TOL), atol=1e-7))), f"linhas_invalidas={int((~np.isclose(qtd, np.floor(qtd + TOL), atol=1e-7)).sum())}")

    if not positivos.empty:
        item = positivos.groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False).agg(
            QTD=("QTD_PRODUZIR_SOLVER", "sum"), LOTE_MIN=("LOTE_MIN", "max"), QTD_EMB=("QTD_EMB", "max"), NEC=("NEC_PCS", "max")
        )
        inv_lote = item[(item["LOTE_MIN"] > TOL) & (item["QTD"] + TOL < item["LOTE_MIN"])]
        add("LOTE_MIN_VALIDO", inv_lote.empty, f"itens_invalidos={len(inv_lote)}")
        mask_emb = item["QTD_EMB"] > TOL
        resto = np.mod(item.loc[mask_emb, "QTD"], item.loc[mask_emb, "QTD_EMB"])
        inv_emb = item.loc[mask_emb & (~np.isclose(np.mod(item["QTD"], item["QTD_EMB"].where(item["QTD_EMB"] > TOL, 1)), 0.0, atol=1e-7))]
        add("QTD_EMB_MULTIPLO_VALIDO", inv_emb.empty, f"itens_invalidos={len(inv_emb)}")
        inv_nec = item[item["QTD"] > item["NEC"] + TOL]
        add("PRODUCAO_MENOR_IGUAL_NEC", inv_nec.empty, f"itens_invalidos={len(inv_nec)}")
    else:
        add("LOTE_MIN_VALIDO", True, "sem_producao_positiva")
        add("QTD_EMB_MULTIPLO_VALIDO", True, "sem_producao_positiva")
        add("PRODUCAO_MENOR_IGUAL_NEC", True, "sem_producao_positiva")

    uso_rec = plano.groupby("ALOC_REC", dropna=True)["HR_PRODUZIR_SOLVER"].sum()
    cap_rec = solver["cap_rec"].set_index("ALOC_REC")["HOR_REC"] if not solver["cap_rec"].empty else pd.Series(dtype=float)
    comp_rec = pd.concat([uso_rec.rename("USO"), cap_rec.rename("CAP")], axis=1).fillna(0.0)
    inv_rec = comp_rec[comp_rec["USO"] > comp_rec["CAP"] + TOL]
    add("CAPACIDADE_ALOC_REC", inv_rec.empty, f"recursos_estourados={len(inv_rec)}; max_estouro={(inv_rec['USO']-inv_rec['CAP']).max() if not inv_rec.empty else 0:.8f}")

    uso_fer = plano.groupby("COD_FER_UNID", dropna=True)["HR_PRODUZIR_SOLVER"].sum()
    cap_fer = solver["cap_fer"].set_index("COD_FER_UNID")["HOR_FER"] if not solver["cap_fer"].empty else pd.Series(dtype=float)
    comp_fer = pd.concat([uso_fer.rename("USO"), cap_fer.rename("CAP")], axis=1).fillna(0.0)
    inv_fer = comp_fer[comp_fer["USO"] > comp_fer["CAP"] + TOL]
    add("CAPACIDADE_COD_FER_UNID", inv_fer.empty, f"ferramentas_estouradas={len(inv_fer)}; max_estouro={(inv_fer['USO']-inv_fer['CAP']).max() if not inv_fer.empty else 0:.8f}")

    add("MATRIZ_CONVERGIU", bool(validacao_matriz.get("MATRIZ_CONVERGIU", False)), f"gap_residual_pcs={validacao_matriz.get('MATRIZ_GAP_RESIDUAL_PCS', 0):.6f}; reconciliacoes={validacao_matriz.get('MATRIZ_RECONCILIACOES', 0)}")

    if auditoria_matriz is not None and not auditoria_matriz.empty:
        dif = (to_num(auditoria_matriz["VALOR_ANTES_PCS"]) - to_num(auditoria_matriz["CORTE_PCS"]) - to_num(auditoria_matriz["VALOR_DEPOIS_PCS"])).abs()
        add("AUDITORIA_EVENTO_FECHA", bool((dif <= 1e-6).all()), f"eventos_divergentes={int((dif > 1e-6).sum())}")
        proibidos = set(auditoria_matriz["COMPONENTE_CORTE"].astype(str)) - {"C_ARR", "C_AT", "PREV_VEN", "PREV_PROX", "EST_SEG"}
        add("SOMENTE_COMPONENTES_CORTAVEIS", not proibidos, f"proibidos={sorted(proibidos)}")
        criterios = auditoria_matriz["CRITERIO_MATRIZ"].astype(str).str.replace(r"^\d+_", "", regex=True)
        permitidos = {f"{m}|{c}" for _, m, c in MATRIZ_CRITERIOS}
        invalidos = sorted(set(criterios) - permitidos)
        add("CRITERIOS_MATRIZ_VALIDOS", not invalidos, f"invalidos={invalidos[:10]}")
    else:
        add("AUDITORIA_EVENTO_FECHA", True, "nenhum_corte_necessario")
        add("SOMENTE_COMPONENTES_CORTAVEIS", True, "nenhum_corte_necessario")
        add("CRITERIOS_MATRIZ_VALIDOS", True, "nenhum_corte_necessario")

    for ori, atual in [("ORI_C_ARR", "C_ARR"), ("ORI_C_AT", "C_AT"), ("ORI_PREV_VEN", "PREV_VEN"), ("ORI_PREV_PROX", "PREV_PROX"), ("ORI_EST_SEG", "EST_SEG")]:
        if ori in plano.columns and atual in plano.columns:
            inv = plano[to_num(plano[atual]) > to_num(plano[ori]) + TOL]
            add(f"{ori}_IMUTAVEL_REFERENCIA", inv.empty, f"linhas_final_maior_original={len(inv)}")

    validacao = pd.DataFrame(registros)
    if falhas:
        raise RuntimeError("VALIDACAO_MES_FALHOU | " + " | ".join(falhas[:12]))
    return validacao

print("[07/10] OK - Validacoes criticas preparadas")


In [ ]:
# ============================================================
# ETAPA 9 - ENCADEAMENTO MENSAL E EXECUCAO
# OBJETIVO: processar M1 primeiro; somente com M1 validado avançar M2 -> M3 -> M4.
# ENTRADAS: bd_entrada_completa.
# PODE ALTERAR: EST_IN e C_ARR dos meses seguintes pelo estado transportado.
# DEVE PRESERVAR: chave COD_PROD + UNID_FAT e estado de itens temporariamente ausentes.
# SAIDAS: plano_producao, auditoria_matriz_cortes e validacoes internas.
# VALIDACAO ESPERADA: transporte mensal reconciliado e cada mes PASSOU antes do seguinte.
# ============================================================

def _chave_estado(cod_prod, unid_fat):
    # Chave oficial alinhada com o cliente: estoque/carteira fecham e reabrem por COD_PROD + UNID_FAT.
    return (str(cod_prod).strip(), str(unid_fat).strip())


def _base_item_periodo(bd_periodo):
    cols_valor = [
        'LTP_EST_INI_PCS','LTP_EST_TRANS_PCS','ORI_TOT_PCS','TRIANG_TOT_PCS',
        'LTP_SALDO_PREV_PCS','LTP_CART_MES_ATUAL','LTP_CART_ARR_MES_ANT'
    ]
    base = bd_periodo.copy()
    base['MES_REF'] = pd.to_datetime(base['MES_REF'], errors='coerce')
    base['COD_PROD'] = normalizar_codigo_serie(base['COD_PROD'])
    base['UNID_FAT'] = base['UNID_FAT'].astype('string').fillna('').str.strip()
    for c in cols_valor:
        if c not in base.columns:
            base[c] = 0.0
        base[c] = to_num(base[c])
    chave = ['MES_REF','ID_PROD_UNID_FAT']
    agg = {c:'max' for c in cols_valor}
    agg.update({c:'first' for c in ['COD_PROD','UNID_PROD','UNID_FAT'] if c in base.columns})
    return base.groupby(chave, as_index=False).agg(agg)


def _calcular_estado_proximo_periodo(bd_periodo, resultado_periodo):
    base = _base_item_periodo(bd_periodo)
    plano = resultado_periodo['plano_producao'].copy()
    if plano.empty:
        prod = pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','PRODUCAO_MES'])
    else:
        plano['MES_REF'] = pd.to_datetime(plano['MES_REF'], errors='coerce')
        plano['QTD_PRODUZIR_SOLVER'] = to_num(plano['QTD_PRODUZIR_SOLVER'])
        prod = plano.groupby(['MES_REF','ID_PROD_UNID_FAT'], as_index=False).agg(PRODUCAO_MES=('QTD_PRODUZIR_SOLVER','sum'))

    dem = resultado_periodo.get('demanda_final_m1', pd.DataFrame()).copy()
    if dem.empty or 'NEC_BRUTA_FINAL' not in dem.columns:
        estrut = pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','DEMANDA_ESTRUTURAL_MES'])
    else:
        dem['MES_REF'] = pd.to_datetime(dem['MES_REF'], errors='coerce')
        dem['NEC_BRUTA_FINAL'] = to_num(dem['NEC_BRUTA_FINAL'])
        estrut = dem.groupby(['MES_REF','ID_PROD_UNID_FAT'], as_index=False).agg(DEMANDA_ESTRUTURAL_MES=('NEC_BRUTA_FINAL','max'))

    saldo = base.merge(prod, on=['MES_REF','ID_PROD_UNID_FAT'], how='left').merge(estrut, on=['MES_REF','ID_PROD_UNID_FAT'], how='left')
    saldo[['PRODUCAO_MES','DEMANDA_ESTRUTURAL_MES']] = saldo[['PRODUCAO_MES','DEMANDA_ESTRUTURAL_MES']].fillna(0.0)
    saldo['SALDO_PROX_PERIODO'] = (
        saldo['LTP_EST_INI_PCS'] + saldo['LTP_EST_TRANS_PCS'] + saldo['ORI_TOT_PCS'] + saldo['TRIANG_TOT_PCS']
        + saldo['PRODUCAO_MES'] - saldo['LTP_SALDO_PREV_PCS'] - saldo['LTP_CART_MES_ATUAL']
        - saldo['LTP_CART_ARR_MES_ANT'] - saldo['DEMANDA_ESTRUTURAL_MES']
    )
    saldo['ESTOQUE_INICIAL_PROX_PERIODO'] = saldo['SALDO_PROX_PERIODO'].clip(lower=0.0)
    saldo['CARTEIRA_ARRASTE_PROX_PERIODO'] = (-saldo['SALDO_PROX_PERIODO']).clip(lower=0.0)
    saldo['CHAVE_ESTADO'] = [_chave_estado(c, u) for c, u in zip(saldo['COD_PROD'], saldo['UNID_FAT'])]
    return saldo


def _aplicar_estado_anterior(bd_periodo, estado_itens):
    out = bd_periodo.copy()
    if 'LTP_EST_INI_PCS' not in out.columns:
        out['LTP_EST_INI_PCS'] = 0.0
    if 'LTP_CART_ARR_MES_ANT' not in out.columns:
        out['LTP_CART_ARR_MES_ANT'] = 0.0
    out['COD_PROD'] = normalizar_codigo_serie(out['COD_PROD'])
    out['UNID_FAT'] = out['UNID_FAT'].astype('string').fillna('').str.strip()
    chaves = [_chave_estado(c, u) for c, u in zip(out['COD_PROD'], out['UNID_FAT'])]
    # Itens novos no horizonte, sem estado anterior, entram com zero projetado; itens existentes usam o fechamento anterior.
    out['LTP_EST_INI_PCS'] = [float(estado_itens[k]['ESTOQUE']) if k in estado_itens else 0.0 for k in chaves]
    out['LTP_CART_ARR_MES_ANT'] = [float(estado_itens[k]['ARRASTE']) if k in estado_itens else 0.0 for k in chaves]
    out['_CHAVE_ESTADO'] = chaves
    # NEC e campos derivados precisam ser recalculados com o novo estoque inicial e novo arraste.
    out = out.drop(columns=[c for c in ['NEC_PCS','NEC_ESTRUTURA_PAI_PCS','NEC_DIRETA_BRUTA_PCS'] if c in out.columns], errors='ignore')
    return out


def _auditar_transporte_estado(bd_periodo, estado_anterior, mes_ref):
    # Compara, antes do Solver do mês, o fechamento anterior com o valor efetivamente injetado no mês corrente.
    if not estado_anterior:
        return pd.DataFrame()
    atual = bd_periodo.copy()
    atual['COD_PROD'] = normalizar_codigo_serie(atual['COD_PROD'])
    atual['UNID_FAT'] = atual['UNID_FAT'].astype('string').fillna('').str.strip()
    atual['CHAVE_ESTADO'] = [_chave_estado(c, u) for c, u in zip(atual['COD_PROD'], atual['UNID_FAT'])]
    atual = atual.groupby('CHAVE_ESTADO', as_index=False).agg(
        COD_PROD=('COD_PROD','first'), UNID_FAT=('UNID_FAT','first'),
        ESTOQUE_INICIAL_ATUAL=('LTP_EST_INI_PCS','max'), CARTEIRA_ARR_ATUAL=('LTP_CART_ARR_MES_ANT','max')
    )
    registros=[]
    for r in atual.itertuples(index=False):
        k=r.CHAVE_ESTADO
        if k not in estado_anterior:
            continue
        esp=estado_anterior[k]
        dif_est=float(r.ESTOQUE_INICIAL_ATUAL)-float(esp['ESTOQUE'])
        dif_arr=float(r.CARTEIRA_ARR_ATUAL)-float(esp['ARRASTE'])
        registros.append({
            'MES_REF':pd.Timestamp(mes_ref),'COD_PROD':r.COD_PROD,'UNID_FAT':r.UNID_FAT,
            'ESTOQUE_FINAL_MES_ANT':float(esp['ESTOQUE']),'ESTOQUE_INICIAL_MES_ATUAL':float(r.ESTOQUE_INICIAL_ATUAL),
            'DIF_ESTOQUE':dif_est,'CARTEIRA_FINAL_MES_ANT':float(esp['ARRASTE']),
            'CARTEIRA_ARR_MES_ATUAL':float(r.CARTEIRA_ARR_ATUAL),'DIF_CARTEIRA':dif_arr,
            'STATUS':'OK' if abs(dif_est)<=TOL and abs(dif_arr)<=TOL else 'DIVERGENTE'
        })
    audit=pd.DataFrame(registros)
    if not audit.empty and audit['STATUS'].ne('OK').any():
        qtd=int(audit['STATUS'].ne('OK').sum())
        raise RuntimeError(f'ENCADEAMENTO_MENSAL_DIVERGENTE: {qtd} chaves COD_PROD + UNID_FAT nao reconciliaram em {pd.Timestamp(mes_ref):%Y-%m}.')
    return audit


def processar_periodo(bd_periodo):
    bd_mes = calcular_nec_periodo(bd_periodo)
    bd_mes, estrutura_explodida, problemas_estrutura, nec_consolidada = processar_estrutura_periodo(bd_mes)
    bd_mes, nec_consolidada, auditoria_matriz, validacao_matriz = executar_matriz_cortes(bd_mes, estrutura_explodida)
    solver = executar_solver_capacidade(bd_mes)
    plano = montar_plano_producao(bd_mes, solver)
    mes_ref = pd.to_datetime(bd_mes["MES_REF"], errors="coerce").min()
    validacao_periodo = validar_periodo(plano, auditoria_matriz, solver, validacao_matriz, mes_ref)

    demanda_final = nec_consolidada[[c for c in ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "UNID_PROD", "UNID_FAT", "NEC_EXPLODIDA_BRUTA_PCS"] if c in nec_consolidada.columns]].copy()
    if "NEC_EXPLODIDA_BRUTA_PCS" in demanda_final.columns:
        demanda_final = demanda_final.rename(columns={"NEC_EXPLODIDA_BRUTA_PCS": "NEC_BRUTA_FINAL"})
    else:
        demanda_final["NEC_BRUTA_FINAL"] = 0.0
    return {
        "plano_producao": plano,
        "auditoria_matriz_cortes": auditoria_matriz,
        "validacao_periodo": validacao_periodo,
        "demanda_final_m1": demanda_final,
        "problemas_estrutura": problemas_estrutura,
    }

iniciar_etapa("PROCESSAMENTO_MENSAL")
estado_itens = {}
partes_plano = []
partes_auditoria = []
partes_validacao = []
auditoria_encadeamento = []
saldos_periodos = []
tempos_periodos = []

for indice_periodo, mes_ref in enumerate(periodos):
    bd_periodo = bd_entrada_completa.loc[bd_entrada_completa["MES_REF"].eq(mes_ref)].copy().reset_index(drop=True)
    inicio_periodo = time.perf_counter()
    if indice_periodo > 0:
        bd_periodo = _aplicar_estado_anterior(bd_periodo, estado_itens)
        audit = _auditar_transporte_estado(bd_periodo, estado_itens, mes_ref)
        if not audit.empty:
            auditoria_encadeamento.append(audit)

    resultado = processar_periodo(bd_periodo)
    partes_plano.append(resultado["plano_producao"])
    partes_auditoria.append(resultado["auditoria_matriz_cortes"])
    partes_validacao.append(resultado["validacao_periodo"])

    saldo = _calcular_estado_proximo_periodo(bd_periodo, resultado)
    saldos_periodos.append(saldo.copy())
    for r in saldo.itertuples(index=False):
        chave = _chave_estado(r.COD_PROD, r.UNID_FAT)
        estado_itens[chave] = {
            "ESTOQUE": float(r.ESTOQUE_INICIAL_PROX_PERIODO),
            "ARRASTE": float(r.CARTEIRA_ARRASTE_PROX_PERIODO),
        }
    tempos_periodos.append({"MES_REF": mes_ref, "SEGUNDOS": time.perf_counter() - inicio_periodo})
    print(f"       {pd.Timestamp(mes_ref):%Y-%m} - PASSOU ({tempos_periodos[-1]['SEGUNDOS']:.2f}s)")

plano_producao = pd.concat(partes_plano, ignore_index=True, sort=False) if partes_plano else pd.DataFrame()
auditoria_matriz_cortes = pd.concat(partes_auditoria, ignore_index=True, sort=False) if partes_auditoria else pd.DataFrame()
validacao_periodos = pd.concat(partes_validacao, ignore_index=True, sort=False) if partes_validacao else pd.DataFrame()
saldo_encadeamento = pd.concat(saldos_periodos, ignore_index=True, sort=False) if saldos_periodos else pd.DataFrame()
auditoria_encadeamento = pd.concat(auditoria_encadeamento, ignore_index=True, sort=False) if auditoria_encadeamento else pd.DataFrame()

finalizar_etapa("PROCESSAMENTO_MENSAL")
print("[08/10] OK - Todos os periodos processados em sequencia")


In [ ]:
# ============================================================
# ETAPA 10 - EXPORTACAO E FECHAMENTO
# OBJETIVO: exportar somente as duas guias autorizadas e finalizar a execucao.
# ENTRADAS: plano_producao e auditoria_matriz_cortes.
# PODE ALTERAR: somente formatacao do arquivo final.
# DEVE PRESERVAR: valores calculados.
# SAIDAS: bd_SOLVER_v66_4.xlsx.
# VALIDACAO ESPERADA: somente 03_PLANO_PRODUCAO e 04_AUDITORIA_MATRIZ_CORTES.
# ============================================================

iniciar_etapa("EXPORTACAO")
OUTPUT_SOLVER_FILE.parent.mkdir(parents=True, exist_ok=True)
GUIAS_DADOS = {
    "03_PLANO_PRODUCAO": plano_producao,
    "04_AUDITORIA_MATRIZ_CORTES": auditoria_matriz_cortes,
}
ativas = [nome for nome, ativo in GUIAS_EXPORTAR.items() if ativo]
if ativas != ["03_PLANO_PRODUCAO", "04_AUDITORIA_MATRIZ_CORTES"]:
    raise RuntimeError(f"GUIAS_FINAIS_INVALIDAS: {ativas}")

with pd.ExcelWriter(OUTPUT_SOLVER_FILE, engine="openpyxl") as writer:
    for nome in ativas:
        GUIAS_DADOS[nome].to_excel(writer, sheet_name=nome, index=False)

wb = load_workbook(OUTPUT_SOLVER_FILE)
header_fill = PatternFill("solid", fgColor="1F4E78")
header_font = Font(color="FFFFFF", bold=True)
thin_gray = Side(style="thin", color="D9E2F3")
for ws in wb.worksheets:
    ws.freeze_panes = "A2"
    ws.sheet_view.showGridLines = False
    ws.auto_filter.ref = ws.dimensions
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = Border(bottom=thin_gray)
    for idx, cells in enumerate(ws.columns, start=1):
        max_len = max((len(str(c.value)) for c in cells if c.value is not None), default=8)
        ws.column_dimensions[get_column_letter(idx)].width = min(max(max_len + 2, 10), 42)
wb.save(OUTPUT_SOLVER_FILE)

finalizar_etapa("EXPORTACAO")
tempo_total = time.perf_counter() - TEMPO_TOTAL_INICIO
print("[09/10] OK - Excel final gravado")
print(f"[10/10] CONCLUIDO - {OUTPUT_SOLVER_FILE.name} | {tempo_total:.2f}s")
print("Tempos por etapa:")
for nome, segundos in TEMPOS_ETAPAS.items():
    print(f"  {nome}: {segundos:.2f}s")
